<a href="https://colab.research.google.com/github/vikassingh0593/bytemaster_stocks/blob/dev/web_scrapping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Update package lists silently
!apt-get update -qq > /dev/null

# Install OpenJDK 11 (required for Spark)
!apt-get install openjdk-11-jdk-headless -qq > /dev/null

# Download Spark 3.1.1 with Hadoop 3.2
!wget -q http://archive.apache.org/dist/spark/spark-3.1.1/spark-3.1.1-bin-hadoop3.2.tgz

# Extract the downloaded Spark archive
!tar xf spark-3.1.1-bin-hadoop3.2.tgz

# Install the 'findspark' library for easy integration
!pip install -q findspark

# Set environment variables for Java and Spark
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"  # Path to OpenJDK 11
os.environ["SPARK_HOME"] = "/content/spark-3.1.1-bin-hadoop3.2"  # Path to Spark

# Initialize findspark and PySpark
import findspark
findspark.init()

from pyspark.sql import SparkSession
# Create a Spark session
spark = SparkSession.builder.master("local[*]").appName("MySparkApp").getOrCreate()

# Verify the Spark session
print("Spark version:", spark.version)  # Print the Spark version to confirm setup


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Spark version: 3.1.1


In [2]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True) # Property used to format output tables better

import pandas as pd
import requests
from bs4 import BeautifulSoup
from pyspark.sql.functions import col, when, last, monotonically_increasing_id, lag, lead, coalesce, lit
from pyspark.sql.window import Window
from functools import reduce
from pyspark.sql import DataFrame

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import lower
from pyspark.sql import functions as F

In [3]:
# Install PySpark and yfinance
!pip install pyspark yfinance

import yfinance as yf
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit

import os
from google.colab import drive
!pip install yfinance
# !pip install --upgrade numpy
from yfinance import Ticker
spark.conf.set("spark.sql.debug.maxToStringFields", 1000) # Or a higher value as needed

In [4]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [5]:
def rd_fn(Number, Qtr, Yer):

  url = f"https://www.bseindia.com/corporates/shpPublicShareholder.aspx?scripcd={Number}&qtrid=121.00&QtrName={Qtr}%20{Yer}"

  # Define headers to mimic a real browser request
  headers = {
      "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
  }

  # Define cookies if required
  cookies = {
      "cookie_name": "cookie_value"
  }

  # Send a GET request to the URL with headers and cookies
  response = requests.get(url, headers=headers, cookies=cookies)

  soup = BeautifulSoup(response.content, "html.parser")

  rows = soup.find_all('tr')

  for td in rows[7].find_all('td'):
    title = td.get_text().strip()

  data_lst = []
  for i in range(17, len(rows)):
    try:
      text_list = [td.get_text().strip() for td in rows[i].find_all('td')[1:8]]
      if text_list[0]=='B1) Institutions':
        append_flag = True
      if append_flag:
        data_lst.append(text_list)
      # i+=1
    except IndexError:
      append_flag = False

  # for i in data_lst:
  #   print(len(i), i)

  columns = ['CategoryNameoftheShareholders', 'NoOfShareholder', 'NoOfFullyPaidShares', 'Blank_1', 'Blank_2', 'TotalNoSharesHeld', 'ShareholdingPerc']
  pandas_df = pd.DataFrame(data_lst, columns=columns)

  # Replace null values and empty spaces with 0
  # pandas_df.replace(to_replace=['', ' ', None, pd.NA], value=0, inplace=True)

  # Convert data types
  pandas_df['NoOfShareholder'] = pandas_df['NoOfShareholder'].astype(int)
  pandas_df['ShareholdingPerc'] = pandas_df['ShareholdingPerc'].astype(float)

  # Filter rows where NoOfShareholder is equal to 1
  pandas_df = pandas_df[pandas_df['NoOfShareholder'] == 1]

  # Drop the original Blank_1 and Blank_2 columns
  pandas_df.drop(columns=['Blank_1', 'Blank_2', 'NoOfShareholder', 'TotalNoSharesHeld'], inplace=True)
  pandas_df['Number'] = Number
  pandas_df['Title'] = title
  pandas_df['Qtr'] = Qtr
  pandas_df['Year'] = Yer

  return pandas_df

In [11]:
Number = 725420

Qtr = "March"
Yer = "2024"
df = rd_fn(Number, Qtr, Yer)
df

IndexError: list index out of range

In [ ]:
import os

# Base directory in Google Drive
base_dir = "/content/drive/My Drive/bytemaster_stocks"

# df_dct = {}
err_lst = []

for i in lst[3:]:
    try:
        Qtr = "March"
        Yer = "2024"
        df = rd_fn(i, Qtr, Yer)
        # df_dct[i] = df

        # Create folder structure
        year_folder = os.path.join(base_dir, Yer)
        qtr_folder = os.path.join(year_folder, Qtr)

        # Create directories if they don't exist
        os.makedirs(qtr_folder, exist_ok=True)

        # Define the file path
        file_path = os.path.join(qtr_folder, f"{i}.csv")

        # Save DataFrame to CSV
        df.to_csv(file_path, index=False)
        print(f"File saved: {file_path}")

    except Exception as e:
        print(f"Error processing {i}: {e}")
        err_lst.append(i)


In [20]:
err_lst

[]

In [ ]:
# creating combined file
from google.colab import drive
import pandas as pd
import os

# Mount Google Drive
# drive.mount('/content/drive')

# Path to the folder containing CSV files
folder_path = '/content/drive/My Drive/bytemaster_stocks/2024/March'

# Read all CSV files in the folder
dataframes = []
for file in os.listdir(folder_path):
    if file.endswith('.csv'):  # Only process CSV files
        file_path = os.path.join(folder_path, file)
        df = pd.read_csv(file_path)
        dataframes.append(df)

# Example: Concatenate all DataFrames into one (if applicable)
final_df = pd.concat(dataframes, ignore_index=True)

# Save the combined DataFrame to the same folder
output_file_path = os.path.join(folder_path, 'combined_data.csv')
final_df.to_csv(output_file_path, index=False)

print(f"Combined CSV file saved at: {output_file_path}")
print(final_df.head())  # Preview the combined DataFrame


In [8]:
lst = [
500002,
500003,
500008,
500009,
500012,
500014,
500016,
500020,
500023,
500027,
500028,
500031,
500032,
500033,
500034,
500038,
500039,
500040,
500041,
500042,
500043,
500048,
500049,
500052,
500058,
500059,
500060,
500067,
500068,
500069,
500074,
500078,
500083,
500084,
500085,
500086,
500087,
500089,
500092,
500093,
500096,
500097,
500101,
500103,
500104,
500106,
500108,
500109,
500110,
500112,
500113,
500114,
500116,
500117,
500119,
500120,
500123,
500124,
500125,
500126,
500128,
500133,
500135,
500136,
500142,
500143,
500144,
500147,
500148,
500150,
500153,
500159,
500160,
500163,
500164,
500165,
500166,
500168,
500170,
500171,
500173,
500174,
500179,
500180,
500182,
500183,
500184,
500185,
500186,
500187,
500188,
500189,
500191,
500192,
500193,
500199,
500201,
500202,
500206,
500207,
500209,
500210,
500213,
500214,
500215,
500219,
500220,
500223,
500227,
500228,
500231,
500233,
500234,
500235,
500236,
500238,
500239,
500240,
500241,
500243,
500245,
500246,
500247,
500248,
500249,
500250,
500251,
500252,
500253,
500257,
500259,
500260,
500262,
500264,
500265,
500266,
500267,
500268,
500270,
500271,
500277,
500279,
500280,
500282,
500284,
500285,
500288,
500290,
500292,
500294,
500295,
500296,
500298,
500300,
500302,
500304,
500306,
500307,
500312,
500313,
500314,
500317,
500319,
500322,
500325,
500327,
500330,
500331,
500333,
500335,
500336,
500337,
500338,
500339,
500342,
500343,
500346,
500350,
500354,
500355,
500356,
500357,
500360,
500365,
500367,
500368,
500370,
500378,
500380,
500387,
500388,
500390,
500400,
500402,
500403,
500404,
500405,
500407,
500408,
500410,
500411,
500412,
500413,
500414,
500418,
500420,
500421,
500422,
500425,
500426,
500429,
500439,
500440,
500444,
500449,
500450,
500456,
500458,
500459,
500460,
500463,
500464,
500467,
500469,
500470,
500472,
500477,
500480,
500483,
500488,
500490,
500493,
500495,
500500,
500510,
500520,
500530,
500547,
500550,
500570,
500575,
500620,
500645,
500650,
500655,
500660,
500670,
500672,
500674,
500680,
500690,
500696,
500710,
500730,
500770,
500777,
500780,
500790,
500800,
500820,
500825,
500830,
500840,
500850,
500870,
500875,
500877,
500878,
500890,
500940,
501110,
501111,
501144,
501148,
501150,
501261,
501270,
501295,
501298,
501301,
501311,
501314,
501343,
501370,
501386,
501391,
501421,
501423,
501425,
501430,
501455,
501477,
501479,
501622,
501630,
501700,
501831,
501833,
501848,
502015,
502090,
502133,
502137,
502157,
502168,
502175,
502180,
502219,
502250,
502281,
502294,
502330,
502355,
502420,
502445,
502448,
502450,
502587,
502589,
502820,
502850,
502865,
502873,
502893,
502901,
502933,
502937,
502958,
502986,
503031,
503092,
503100,
503101,
503127,
503162,
503169,
503229,
503310,
503349,
503622,
503624,
503626,
503635,
503639,
503641,
503657,
503659,
503663,
503669,
503671,
503675,
503681,
503685,
503689,
503696,
503722,
503772,
503776,
503804,
503806,
503811,
503816,
503863,
503893,
503960,
504000,
504028,
504036,
504058,
504067,
504076,
504080,
504084,
504092,
504093,
504112,
504132,
504176,
504180,
504212,
504220,
504240,
504258,
504273,
504286,
504340,
504341,
504346,
504351,
504356,
504365,
504375,
504378,
504380,
504392,
504397,
504605,
504614,
504646,
504648,
504731,
504741,
504786,
504810,
504840,
504879,
504882,
504903,
504908,
504918,
504959,
504961,
504973,
504988,
504998,
505010,
505032,
505036,
505075,
505141,
505160,
505163,
505192,
505196,
505200,
505212,
505216,
505232,
505242,
505250,
505255,
505283,
505285,
505299,
505302,
505324,
505336,
505343,
505355,
505358,
505368,
505400,
505412,
505502,
505504,
505509,
505515,
505520,
505523,
505526,
505533,
505537,
505585,
505590,
505594,
505650,
505681,
505685,
505688,
505690,
505693,
505700,
505703,
505710,
505712,
505714,
505720,
505725,
505726,
505729,
505737,
505744,
505750,
505790,
505797,
505800,
505807,
505827,
505840,
505850,
505854,
505872,
505890,
505893,
505978,
506003,
506022,
506024,
506042,
506074,
506076,
506079,
506105,
506109,
506120,
506122,
506128,
506134,
506146,
506161,
506162,
506166,
506178,
506180,
506184,
506186,
506190,
506194,
506196,
506197,
506222,
506235,
506248,
506260,
506261,
506285,
506313,
506365,
506390,
506395,
506401,
506405,
506414,
506480,
506520,
506525,
506528,
506530,
506532,
506543,
506579,
506590,
506597,
506605,
506618,
506640,
506642,
506655,
506680,
506685,
506687,
506690,
506734,
506767,
506808,
506820,
506852,
506854,
506858,
506879,
506906,
506910,
506919,
506935,
506943,
506945,
506947,
506975,
506979,
506981,
507155,
507180,
507205,
507265,
507300,
507315,
507410,
507438,
507474,
507486,
507488,
507490,
507498,
507514,
507515,
507526,
507530,
507543,
507552,
507580,
507598,
507609,
507621,
507645,
507663,
507685,
507690,
507717,
507747,
507753,
507759,
507779,
507785,
507789,
507794,
507808,
507813,
507815,
507817,
507828,
507833,
507836,
507852,
507864,
507872,
507878,
507880,
507910,
507912,
507938,
507944,
507946,
507948,
507952,
507960,
507962,
507966,
507970,
507981,
507987,
507998,
508136,
508486,
508494,
508571,
508664,
508670,
508807,
508814,
508867,
508869,
508875,
508905,
508906,
508918,
508922,
508933,
508941,
508954,
508956,
508961,
508969,
508980,
508989,
508993,
509003,
509009,
509015,
509020,
509026,
509038,
509040,
509046,
509048,
509051,
509053,
509055,
509073,
509079,
509084,
509152,
509162,
509196,
509220,
509243,
509423,
509438,
509449,
509470,
509472,
509480,
509486,
509488,
509496,
509525,
509546,
509557,
509563,
509567,
509597,
509631,
509635,
509650,
509675,
509692,
509709,
509715,
509732,
509760,
509782,
509820,
509835,
509845,
509870,
509874,
509887,
509895,
509910,
509917,
509930,
509945,
509960,
509966,
510245,
511000,
511012,
511016,
511018,
511034,
511048,
511060,
511066,
511074,
511076,
511092,
511096,
511108,
511110,
511116,
511122,
511131,
511147,
511153,
511176,
511185,
511187,
511194,
511196,
511200,
511208,
511218,
511243,
511246,
511254,
511260,
511333,
511355,
511359,
511377,
511391,
511401,
511411,
511413,
511431,
511441,
511447,
511451,
511463,
511473,
511493,
511501,
511505,
511507,
511509,
511523,
511525,
511533,
511535,
511543,
511549,
511551,
511557,
511559,
511563,
511571,
511585,
511589,
511593,
511601,
511605,
511609,
511611,
511626,
511628,
511630,
511634,
511644,
511654,
511658,
511664,
511672,
511676,
511688,
511692,
511696,
511700,
511702,
511710,
511712,
511714,
511724,
511726,
511728,
511738,
511740,
511742,
511754,
511756,
511758,
511760,
511764,
511766,
511768,
512004,
512008,
512014,
512018,
512020,
512022,
512024,
512025,
512026,
512036,
512038,
512047,
512048,
512060,
512062,
512063,
512064,
512065,
512068,
512070,
512091,
512093,
512097,
512099,
512101,
512103,
512115,
512117,
512131,
512147,
512149,
512153,
512157,
512161,
512165,
512169,
512175,
512179,
512195,
512197,
512213,
512215,
512217,
512221,
512229,
512237,
512245,
512247,
512257,
512261,
512267,
512271,
512277,
512279,
512291,
512296,
512297,
512301,
512303,
512329,
512341,
512344,
512345,
512367,
512377,
512379,
512381,
512393,
512399,
512404,
512408,
512415,
512425,
512431,
512433,
512437,
512441,
512443,
512445,
512453,
512455,
512461,
512463,
512477,
512479,
512481,
512485,
512489,
512493,
512499,
512505,
512511,
512519,
512527,
512529,
512531,
512553,
512559,
512565,
512573,
512587,
512589,
512591,
512595,
512597,
512599,
512604,
512608,
512618,
512624,
512626,
512634,
513005,
513023,
513039,
513043,
513059,
513063,
513097,
513108,
513117,
513119,
513121,
513149,
513173,
513228,
513250,
513252,
513262,
513269,
513303,
513307,
513309,
513335,
513337,
513349,
513353,
513361,
513369,
513375,
513377,
513397,
513401,
513403,
513418,
513422,
513430,
513436,
513452,
513456,
513460,
513472,
513488,
513496,
513498,
513502,
513507,
513509,
513511,
513513,
513515,
513517,
513519,
513528,
513532,
513536,
513540,
513548,
513554,
513566,
513575,
513579,
513599,
513629,
513642,
513683,
513687,
513693,
513699,
513709,
513713,
513721,
513729,
514010,
514028,
514030,
514036,
514043,
514045,
514060,
514087,
514113,
514128,
514138,
514140,
514142,
514162,
514165,
514167,
514171,
514175,
514183,
514197,
514211,
514223,
514234,
514236,
514238,
514240,
514248,
514260,
514264,
514266,
514272,
514274,
514280,
514286,
514300,
514302,
514312,
514316,
514318,
514322,
514324,
514326,
514330,
514332,
514348,
514354,
514358,
514360,
514378,
514386,
514400,
514402,
514418,
514428,
514442,
514448,
514450,
514454,
514460,
514470,
515008,
515018,
515030,
515037,
515043,
515055,
515059,
515085,
515093,
515127,
515147,
516003,
516016,
516020,
516022,
516030,
516032,
516038,
516062,
516064,
516072,
516078,
516082,
516092,
516096,
516098,
516106,
516108,
516110,
517015,
517035,
517041,
517044,
517059,
517063,
517096,
517119,
517146,
517166,
517168,
517170,
517172,
517174,
517201,
517206,
517214,
517236,
517238,
517246,
517258,
517271,
517273,
517288,
517300,
517334,
517344,
517354,
517356,
517360,
517370,
517372,
517380,
517385,
517393,
517397,
517399,
517411,
517415,
517417,
517421,
517423,
517429,
517431,
517437,
517447,
517449,
517467,
517477,
517494,
517498,
517500,
517506,
517514,
517522,
517530,
517536,
517544,
517546,
517548,
517554,
517556,
517562,
517569,
518011,
518017,
518075,
518091,
519003,
519014,
519031,
519064,
519091,
519097,
519105,
519126,
519136,
519152,
519156,
519174,
519183,
519191,
519216,
519224,
519234,
519238,
519242,
519262,
519285,
519287,
519295,
519299,
519307,
519331,
519353,
519359,
519367,
519383,
519397,
519413,
519415,
519421,
519439,
519455,
519457,
519463,
519471,
519475,
519477,
519483,
519494,
519500,
519506,
519528,
519532,
519552,
519566,
519574,
519600,
519602,
519604,
519606,
519612,
520008,
520021,
520043,
520051,
520056,
520057,
520059,
520066,
520073,
520075,
520081,
520086,
520111,
520113,
520119,
520121,
520123,
520127,
520131,
520141,
520151,
520155,
521003,
521005,
521014,
521016,
521018,
521034,
521036,
521048,
521054,
521062,
521064,
521068,
521070,
521080,
521097,
521105,
521109,
521113,
521131,
521133,
521137,
521141,
521149,
521151,
521161,
521163,
521178,
521180,
521188,
521194,
521200,
521206,
521216,
521220,
521222,
521226,
521228,
521232,
521234,
521238,
521240,
521242,
521244,
521246,
521248,
522001,
522004,
522005,
522014,
522017,
522027,
522029,
522034,
522036,
522064,
522073,
522074,
522091,
522101,
522105,
522108,
522113,
522122,
522152,
522163,
522165,
522171,
522183,
522195,
522205,
522207,
522209,
522215,
522217,
522229,
522231,
522235,
522237,
522241,
522249,
522251,
522257,
522261,
522267,
522273,
522275,
522281,
522285,
522287,
522289,
522292,
522294,
522295,
522650,
523007,
523011,
523019,
523021,
523023,
523025,
523054,
523062,
523100,
523105,
523113,
523116,
523120,
523127,
523144,
523151,
523160,
523186,
523204,
523207,
523222,
523229,
523232,
523242,
523248,
523260,
523261,
523269,
523277,
523283,
523289,
523301,
523309,
523315,
523319,
523323,
523329,
523343,
523367,
523369,
523371,
523373,
523384,
523385,
523391,
523395,
523398,
523405,
523411,
523419,
523425,
523445,
523457,
523465,
523467,
523475,
523483,
523489,
523519,
523537,
523539,
523550,
523558,
523566,
523574,
523586,
523594,
523598,
523606,
523610,
523618,
523620,
523628,
523630,
523638,
523642,
523648,
523650,
523652,
523660,
523672,
523676,
523694,
523696,
523704,
523708,
523710,
523712,
523716,
523722,
523732,
523736,
523752,
523754,
523782,
523790,
523792,
523826,
523828,
523832,
523836,
523838,
523840,
523842,
523844,
523850,
523888,
523896,
524000,
524013,
524019,
524031,
524037,
524038,
524046,
524051,
524075,
524080,
524091,
524109,
524129,
524136,
524156,
524164,
524174,
524200,
524202,
524204,
524208,
524210,
524212,
524218,
524226,
524230,
524280,
524288,
524314,
524324,
524330,
524332,
524336,
524342,
524348,
524370,
524372,
524394,
524396,
524400,
524404,
524408,
524412,
524414,
524434,
524440,
524444,
524458,
524470,
524480,
524488,
524494,
524500,
524502,
524504,
524506,
524514,
524516,
524518,
524520,
524522,
524534,
524542,
524546,
524558,
524564,
524570,
524572,
524576,
524580,
524582,
524590,
524594,
524598,
524602,
524604,
524606,
524614,
524622,
524624,
524628,
524632,
524634,
524636,
524640,
524642,
524648,
524652,
524654,
524661,
524663,
524667,
524669,
524675,
524687,
524703,
524709,
524711,
524715,
524717,
524723,
524727,
524731,
524735,
524742,
524743,
524748,
524752,
524768,
524774,
524790,
524804,
524808,
524816,
524818,
524820,
524824,
524828,
526001,
526025,
526027,
526043,
526073,
526081,
526095,
526113,
526115,
526117,
526125,
526133,
526137,
526139,
526143,
526159,
526161,
526169,
526173,
526179,
526187,
526193,
526211,
526217,
526225,
526227,
526231,
526237,
526241,
526247,
526251,
526263,
526269,
526299,
526301,
526315,
526325,
526335,
526345,
526349,
526355,
526365,
526367,
526371,
526373,
526381,
526397,
526407,
526409,
526415,
526423,
526431,
526433,
526435,
526439,
526441,
526443,
526445,
526468,
526471,
526473,
526479,
526481,
526488,
526492,
526494,
526500,
526506,
526508,
526519,
526521,
526525,
526532,
526544,
526546,
526550,
526568,
526570,
526574,
526576,
526582,
526586,
526588,
526596,
526604,
526608,
526612,
526614,
526616,
526622,
526628,
526638,
526640,
526642,
526650,
526654,
526662,
526666,
526668,
526675,
526677,
526687,
526703,
526705,
526709,
526711,
526717,
526721,
526723,
526725,
526727,
526729,
526731,
526739,
526747,
526751,
526755,
526761,
526773,
526775,
526783,
526795,
526797,
526799,
526807,
526813,
526817,
526821,
526823,
526829,
526839,
526841,
526847,
526849,
526851,
526853,
526859,
526861,
526865,
526869,
526871,
526877,
526881,
526885,
526891,
526899,
526901,
526905,
526921,
526931,
526935,
526945,
526947,
526951,
526953,
526961,
526965,
526967,
526971,
526977,
526981,
526983,
526987,
527001,
527005,
530001,
530005,
530007,
530011,
530017,
530019,
530023,
530025,
530027,
530035,
530037,
530043,
530045,
530053,
530055,
530057,
530063,
530065,
530067,
530073,
530075,
530077,
530079,
530095,
530109,
530111,
530117,
530119,
530125,
530127,
530129,
530131,
530133,
530135,
530139,
530141,
530145,
530151,
530161,
530163,
530167,
530169,
530171,
530173,
530175,
530179,
530185,
530187,
530197,
530199,
530201,
530213,
530215,
530231,
530233,
530235,
530239,
530245,
530249,
530251,
530253,
530255,
530259,
530263,
530265,
530267,
530281,
530289,
530291,
530299,
530305,
530307,
530309,
530313,
530315,
530317,
530331,
530341,
530343,
530355,
530357,
530361,
530363,
530365,
530367,
530369,
530377,
530393,
530401,
530405,
530407,
530419,
530421,
530427,
530429,
530431,
530433,
530439,
530443,
530445,
530449,
530457,
530459,
530461,
530469,
530475,
530477,
530495,
530499,
530517,
530521,
530525,
530533,
530537,
530545,
530547,
530549,
530555,
530557,
530565,
530571,
530577,
530579,
530581,
530585,
530589,
530595,
530601,
530609,
530611,
530615,
530617,
530621,
530627,
530643,
530655,
530663,
530665,
530669,
530675,
530677,
530689,
530695,
530697,
530699,
530705,
530709,
530711,
530713,
530715,
530723,
530733,
530735,
530741,
530747,
530755,
530759,
530765,
530779,
530787,
530789,
530795,
530797,
530799,
530803,
530805,
530809,
530813,
530821,
530825,
530829,
530839,
530843,
530845,
530853,
530855,
530871,
530879,
530881,
530883,
530897,
530899,
530907,
530909,
530917,
530919,
530925,
530927,
530929,
530931,
530943,
530951,
530953,
530959,
530961,
530965,
530973,
530977,
530979,
530991,
530997,
530999,
531003,
531017,
531025,
531027,
531035,
531039,
531041,
531043,
531049,
531051,
531065,
531067,
531069,
531080,
531082,
531083,
531091,
531092,
531099,
531109,
531111,
531112,
531119,
531120,
531127,
531129,
531137,
531144,
531146,
531147,
531153,
531155,
531156,
531157,
531158,
531161,
531162,
531163,
531168,
531169,
531173,
531175,
531176,
531178,
531179,
531199,
531201,
531203,
531205,
531209,
531210,
531212,
531213,
531215,
531216,
531219,
531221,
531223,
531225,
531227,
531228,
531233,
531234,
531235,
531237,
531240,
531241,
531246,
531253,
531254,
531255,
531257,
531259,
531260,
531266,
531268,
531272,
531273,
531278,
531279,
531280,
531281,
531283,
531287,
531288,
531289,
531297,
531300,
531304,
531306,
531307,
531310,
531314,
531322,
531323,
531324,
531328,
531334,
531335,
531337,
531338,
531340,
531341,
531344,
531346,
531349,
531352,
531357,
531358,
531359,
531360,
531364,
531370,
531373,
531380,
531381,
531387,
531390,
531395,
531396,
531397,
531398,
531399,
531400,
531402,
531406,
531409,
531411,
531412,
531413,
531416,
531417,
531426,
531431,
531432,
531433,
531436,
531437,
531439,
531444,
531449,
531453,
531454,
531456,
531460,
531465,
531471,
531472,
531489,
531494,
531497,
531499,
531500,
531502,
531503,
531505,
531506,
531508,
531509,
531512,
531515,
531518,
531521,
531525,
531529,
531531,
531537,
531539,
531540,
531541,
531543,
531548,
531550,
531552,
531553,
531556,
531569,
531578,
531582,
531583,
531585,
531591,
531592,
531594,
531595,
531599,
531600,
531608,
531609,
531624,
531626,
531628,
531633,
531635,
531637,
531638,
531640,
531642,
531644,
531647,
531651,
531652,
531661,
531667,
531668,
531671,
531672,
531673,
531681,
531688,
531694,
531696,
531716,
531717,
531719,
531723,
531726,
531727,
531735,
531737,
531739,
531743,
531744,
531746,
531752,
531758,
531761,
531762,
531768,
531771,
531778,
531779,
531780,
531784,
531795,
531797,
531802,
531810,
531812,
531813,
531814,
531821,
531822,
531832,
531834,
531841,
531842,
531845,
531846,
531847,
531859,
531861,
531862,
531867,
531869,
531870,
531878,
531885,
531887,
531888,
531889,
531892,
531893,
531900,
531902,
531909,
531910,
531911,
531913,
531918,
531921,
531923,
531925,
531929,
531930,
531931,
531936,
531946,
531950,
531952,
531959,
531960,
531962,
531968,
531971,
531977,
531978,
531979,
531980,
531982,
531991,
531996,
531997,
532001,
532005,
532007,
532011,
532015,
532016,
532019,
532022,
532024,
532029,
532035,
532039,
532041,
532042,
532051,
532053,
532054,
532056,
532057,
532067,
532070,
532078,
532083,
532090,
532092,
532100,
532102,
532105,
532113,
532123,
532124,
532134,
532138,
532140,
532141,
532143,
532144,
532145,
532149,
532150,
532154,
532155,
532156,
532159,
532160,
532162,
532163,
532164,
532167,
532172,
532173,
532174,
532175,
532178,
532180,
532181,
532183,
532187,
532189,
532209,
532210,
532212,
532215,
532216,
532217,
532218,
532219,
532221,
532230,
532234,
532240,
532256,
532259,
532262,
532268,
532271,
532281,
532284,
532285,
532286,
532290,
532296,
532300,
532303,
532304,
532305,
532307,
532309,
532310,
532313,
532315,
532320,
532321,
532323,
532324,
532326,
532329,
532331,
532333,
532334,
532335,
532339,
532340,
532341,
532343,
532344,
532345,
532348,
532349,
532350,
532351,
532354,
532355,
532356,
532357,
532362,
532365,
532366,
532368,
532369,
532370,
532371,
532372,
532373,
532374,
532375,
532376,
532379,
532380,
532382,
532384,
532386,
532387,
532388,
532390,
532392,
532395,
532397,
532398,
532400,
532402,
532404,
532406,
532407,
532408,
532410,
532413,
532416,
532419,
532424,
532425,
532430,
532432,
532435,
532439,
532440,
532443,
532444,
532454,
532455,
532457,
532460,
532461,
532466,
532467,
532468,
532475,
532477,
532478,
532479,
532481,
532482,
532483,
532485,
532486,
532488,
532493,
532497,
532500,
532503,
532504,
532505,
532507,
532508,
532509,
532513,
532514,
532515,
532521,
532522,
532523,
532524,
532525,
532527,
532528,
532529,
532531,
532532,
532538,
532539,
532540,
532541,
532543,
532548,
532553,
532555,
532604,
532605,
532610,
532612,
532613,
532614,
532616,
532617,
532624,
532626,
532627,
532628,
532629,
532630,
532633,
532636,
532637,
532638,
532640,
532641,
532642,
532644,
532645,
532648,
532649,
532650,
532651,
532652,
532654,
532656,
532659,
532661,
532662,
532663,
532666,
532667,
532668,
532670,
532673,
532674,
532676,
532683,
532684,
532686,
532687,
532689,
532694,
532695,
532696,
532698,
532699,
532700,
532701,
532702,
532705,
532707,
532708,
532710,
532712,
532713,
532714,
532716,
532717,
532719,
532720,
532722,
532723,
532725,
532726,
532728,
532729,
532730,
532732,
532733,
532734,
532735,
532737,
532738,
532740,
532741,
532742,
532744,
532745,
532748,
532749,
532754,
532755,
532756,
532757,
532759,
532760,
532761,
532762,
532764,
532767,
532768,
532771,
532772,
532773,
532775,
532776,
532777,
532779,
532780,
532782,
532783,
532784,
532785,
532790,
532794,
532796,
532797,
532798,
532799,
532800,
532801,
532804,
532805,
532806,
532807,
532808,
532809,
532810,
532811,
532812,
532814,
532815,
532817,
532820,
532822,
532825,
532826,
532827,
532828,
532829,
532830,
532832,
532834,
532835,
532839,
532841,
532842,
532843,
532845,
532847,
532848,
532850,
532851,
532853,
532855,
532856,
532859,
532864,
532867,
532868,
532869,
532870,
532872,
532873,
532875,
532878,
532879,
532880,
532884,
532885,
532886,
532888,
532889,
532890,
532891,
532892,
532893,
532894,
532895,
532896,
532898,
532899,
532900,
532904,
532906,
532907,
532911,
532915,
532916,
532918,
532921,
532922,
532923,
532924,
532925,
532926,
532927,
532928,
532929,
532930,
532931,
532932,
532933,
532934,
532935,
532937,
532939,
532940,
532941,
532942,
532944,
532945,
532946,
532947,
532951,
532952,
532953,
532955,
532957,
532960,
532966,
532967,
532974,
532975,
532976,
532977,
532978,
532980,
532983,
532985,
532986,
532987,
532988,
532992,
532993,
532994,
532998,
533001,
533007,
533012,
533014,
533018,
533019,
533022,
533023,
533029,
533033,
533047,
533048,
533056,
533078,
533080,
533088,
533090,
533093,
533095,
533096,
533098,
533101,
533104,
533106,
533108,
533110,
533121,
533122,
533137,
533138,
533146,
533148,
533149,
533150,
533151,
533152,
533155,
533156,
533158,
533160,
533161,
533162,
533163,
533164,
533166,
533168,
533169,
533170,
533177,
533179,
533181,
533189,
533192,
533193,
533202,
533203,
533206,
533208,
533210,
533212,
533217,
533218,
533227,
533229,
533239,
533248,
533252,
533259,
533260,
533261,
533262,
533263,
533267,
533268,
533269,
533270,
533271,
533272,
533273,
533274,
533275,
533278,
533282,
533284,
533285,
533286,
533287,
533289,
533292,
533293,
533294,
533295,
533296,
533298,
533301,
533302,
533303,
533306,
533315,
533316,
533317,
533320,
533326,
533329,
533333,
533336,
533339,
533343,
533344,
533398,
533400,
533407,
533427,
533451,
533452,
533470,
533477,
533482,
533506,
533519,
533520,
533540,
533543,
533552,
533553,
533573,
533576,
533581,
533602,
533608,
533629,
533632,
533638,
533655,
533676,
533704,
533758,
533761,
533896,
533941,
533982,
534060,
534063,
534064,
534076,
534091,
534109,
534139,
534190,
534309,
534312,
534328,
534338,
534369,
534392,
534422,
534425,
534532,
534597,
534598,
534600,
534612,
534615,
534618,
534623,
534639,
534659,
534674,
534675,
534680,
534691,
534708,
534732,
534733,
534741,
534742,
534748,
534755,
534758,
534804,
534809,
534816,
534976,
535136,
535204,
535205,
535267,
535276,
535279,
535322,
535387,
535431,
535458,
535514,
535566,
535601,
535602,
535621,
535647,
535648,
535657,
535667,
535693,
535719,
535730,
535754,
535755,
535789,
535910,
535916,
535917,
535958,
536073,
536264,
536493,
536507,
536659,
536672,
536709,
536710,
536737,
536738,
536773,
536846,
536868,
536960,
536974,
537007,
537008,
537069,
537253,
537259,
537291,
537292,
537326,
537392,
537483,
537524,
537536,
537573,
537582,
537669,
537707,
537708,
537709,
537750,
537766,
537784,
537785,
537800,
537820,
537839,
537985,
538057,
538081,
538092,
538119,
538212,
538268,
538273,
538319,
538351,
538365,
538382,
538395,
538401,
538402,
538422,
538446,
538451,
538452,
538464,
538465,
538476,
538496,
538521,
538539,
538540,
538542,
538546,
538556,
538562,
538563,
538564,
538565,
538567,
538568,
538569,
538576,
538579,
538596,
538597,
538598,
538607,
538609,
538610,
538611,
538634,
538635,
538646,
538647,
538652,
538666,
538667,
538668,
538674,
538683,
538684,
538685,
538706,
538707,
538708,
538713,
538714,
538715,
538716,
538730,
538731,
538732,
538734,
538742,
538765,
538770,
538772,
538777,
538778,
538787,
538788,
538794,
538795,
538817,
538833,
538834,
538835,
538836,
538837,
538838,
538857,
538860,
538862,
538863,
538868,
538874,
538875,
538881,
538882,
538890,
538891,
538894,
538895,
538896,
538902,
538918,
538920,
538921,
538922,
538923,
538926,
538928,
538935,
538942,
538943,
538952,
538961,
538962,
538964,
538965,
538970,
538975,
538979,
538987,
538992,
538993,
539005,
539006,
539011,
539012,
539013,
539016,
539017,
539018,
539026,
539031,
539032,
539040,
539041,
539042,
539043,
539044,
539045,
539046,
539056,
539083,
539090,
539091,
539096,
539097,
539098,
539099,
539110,
539111,
539112,
539113,
539115,
539116,
539117,
539118,
539119,
539120,
539121,
539123,
539124,
539126,
539132,
539141,
539143,
539148,
539149,
539150,
539151,
539167,
539174,
539175,
539176,
539177,
539189,
539190,
539195,
539196,
539198,
539199,
539201,
539206,
539216,
539217,
539218,
539219,
539220,
539221,
539222,
539226,
539227,
539228,
539251,
539252,
539253,
539254,
539255,
539265,
539267,
539268,
539273,
539275,
539276,
539277,
539278,
539287,
539288,
539289,
539290,
539291,
539300,
539301,
539302,
539309,
539310,
539312,
539313,
539314,
539331,
539332,
539334,
539336,
539337,
539346,
539353,
539354,
539359,
539378,
539383,
539384,
539391,
539393,
539398,
539399,
539400,
539401,
539402,
539404,
539405,
539406,
539407,
539408,
539409,
539428,
539434,
539435,
539436,
539437,
539447,
539448,
539449,
539450,
539468,
539469,
539470,
539479,
539480,
539487,
539488,
539492,
539494,
539495,
539506,
539515,
539516,
539517,
539518,
539521,
539522,
539523,
539524,
539526,
539527,
539528,
539533,
539542,
539543,
539544,
539545,
539546,
539551,
539552,
539559,
539560,
539561,
539562,
539574,
539584,
539593,
539594,
539596,
539598,
539599,
539607,
539620,
539621,
539636,
539658,
539659,
539660,
539661,
539662,
539669,
539673,
539678,
539679,
539680,
539681,
539682,
539683,
539686,
539692,
539697,
539724,
539725,
539730,
539742,
539760,
539761,
539762,
539767,
539773,
539784,
539785,
539787,
539788,
539798,
539799,
539800,
539807,
539814,
539819,
539834,
539835,
539837,
539839,
539841,
539843,
539854,
539871,
539872,
539875,
539876,
539883,
539884,
539889,
539894,
539911,
539917,
539921,
539927,
539938,
539939,
539945,
539946,
539947,
539956,
539957,
539963,
539978,
539979,
539980,
539982,
539984,
539985,
539986,
539991,
539992,
539997,
540005,
540006,
540008,
540023,
540025,
540026,
540047,
540048,
540061,
540062,
540063,
540065,
540066,
540072,
540073,
540078,
540079,
540080,
540081,
540082,
540083,
540097,
540108,
540115,
540124,
540125,
540132,
540133,
540134,
540135,
540136,
540143,
540144,
540145,
540146,
540147,
540148,
540151,
540153,
540154,
540159,
540168,
540173,
540174,
540175,
540180,
540181,
540190,
540192,
540198,
540199,
540203,
540204,
540205,
540210,
540212,
540221,
540222,
540243,
540252,
540254,
540259,
540266,
540267,
540268,
540269,
540270,
540293,
540310,
540311,
540318,
540332,
540358,
540359,
540360,
540361,
540366,
540376,
540377,
540386,
540393,
540395,
540396,
540401,
540402,
540403,
540404,
540416,
540425,
540455,
540467,
540468,
540481,
540492,
540497,
540515,
540519,
540530,
540544,
540545,
540550,
540570,
540575,
540590,
540595,
540596,
540602,
540611,
540612,
540613,
540614,
540615,
540642,
540647,
540648,
540649,
540650,
540651,
540652,
540654,
540669,
540673,
540678,
540679,
540680,
540681,
540686,
540691,
540692,
540693,
540694,
540696,
540699,
540700,
540701,
540702,
540703,
540704,
540709,
540710,
540715,
540716,
540717,
540718,
540719,
540724,
540725,
540726,
540727,
540728,
540729,
540730,
540735,
540737,
540738,
540743,
540749,
540750,
540755,
540756,
540757,
540762,
540767,
540768,
540769,
540774,
540775,
540776,
540777,
540782,
540786,
540787,
540788,
540789,
540795,
540796,
540797,
540798,
540809,
540811,
540821,
540824,
540829,
540843,
540850,
540879,
540900,
540901,
540902,
540903,
540904,
540914,
540923,
540935,
540936,
540937,
540938,
540952,
540953,
540954,
540955,
540956,
540961,
540975,
540980,
541005,
541006,
541019,
541083,
541096,
541112,
541133,
541143,
541144,
541152,
541153,
541154,
541161,
541163,
541167,
541178,
541179,
541195,
541196,
541206,
541228,
541233,
541269,
541276,
541299,
541301,
541302,
541303,
541304,
541313,
541336,
541337,
541338,
541347,
541352,
541353,
541358,
541400,
541402,
541403,
541418,
541444,
541445,
541450,
541503,
541540,
541546,
541556,
541557,
541578,
541601,
541633,
541634,
541700,
541701,
541702,
541703,
541729,
541735,
541741,
541770,
541771,
541778,
541799,
541809,
541865,
541890,
541929,
541945,
541956,
541967,
541972,
541973,
541974,
541983,
541988,
542011,
542012,
542013,
542019,
542020,
542025,
542034,
542046,
542057,
542066,
542123,
542131,
542141,
542145,
542146,
542155,
542176,
542206,
542216,
542230,
542231,
542232,
542233,
542248,
542285,
542323,
542332,
542333,
542337,
542351,
542367,
542376,
542377,
542383,
542399,
542437,
542446,
542459,
542460,
542484,
542513,
542544,
542579,
542580,
542592,
542597,
542599,
542627,
542628,
542649,
542650,
542651,
542652,
542654,
542655,
542665,
542666,
542667,
542668,
542669,
542670,
542677,
542678,
542679,
542682,
542684,
542694,
542721,
542724,
542725,
542726,
542728,
542729,
542730,
542747,
542752,
542753,
542758,
542759,
542760,
542770,
542771,
542772,
542773,
542774,
542801,
542802,
542803,
542804,
542805,
542806,
542807,
542808,
542809,
542810,
542811,
542812,
542813,
542814,
542815,
542816,
542817,
542818,
542819,
542820,
542830,
542836,
542837,
542838,
542839,
542840,
542841,
542842,
542843,
542844,
542845,
542846,
542847,
542848,
542849,
542850,
542851,
542852,
542857,
542862,
542863,
542864,
542865,
542866,
542867,
542904,
542905,
542906,
542907,
542911,
542918,
542919,
542920,
542921,
542922,
542924,
542931,
542933,
542934,
542938,
543064,
543065,
543066,
543144,
543145,
543146,
543147,
543148,
543149,
543150,
543151,
543152,
543153,
543154,
543155,
543156,
543168,
543169,
543170,
543171,
543172,
543173,
543174,
543175,
543176,
543177,
543178,
543179,
543180,
543181,
543182,
543183,
543184,
543185,
543186,
543187,
543193,
543207,
543208,
543209,
543210,
543211,
543212,
543213,
543214,
543218,
543219,
543220,
543221,
543223,
543224,
543226,
543227,
543228,
543229,
543230,
543231,
543232,
543233,
543234,
543235,
543236,
543237,
543238,
543239,
543240,
543241,
543242,
543243,
543244,
543245,
543246,
543248,
543249,
543251,
543252,
543253,
543254,
543255,
543256,
543257,
543258,
543259,
543260,
543262,
543263,
543264,
543265,
543266,
543267,
543268,
543270,
543271,
543272,
543273,
543274,
543275,
543276,
543277,
543278,
543279,
543280,
543281,
543283,
543284,
543285,
543286,
543287,
543288,
543291,
543292,
543297,
543298,
543299,
543300,
543305,
543306,
543308,
543309,
543310,
543311,
543312,
543317,
543318,
543319,
543320,
543321,
543322,
543323,
543324,
543325,
543326,
543327,
543328,
543329,
543330,
543331,
543332,
543333,
543334,
543335,
543336,
543341,
543346,
543347,
543348,
543349,
543350,
543352,
543357,
543358,
543363,
543364,
543365,
543366,
543367,
543372,
543373,
543374,
543375,
543376,
543377,
543383,
543384,
543386,
543387,
543388,
543389,
543390,
543391,
543396,
543397,
543398,
543399,
543400,
543401,
543410,
543411,
543412,
543413,
543414,
543415,
543416,
543417,
543419,
543420,
543425,
543426,
543427,
543428,
543433,
543434,
543435,
543437,
543438,
543439,
543440,
543441,
543442,
543444,
543449,
543450,
543451,
543453,
543454,
543458,
543460,
543461,
543462,
543463,
543464,
543465,
543470,
543472,
543473,
543474,
543475,
543481,
543482,
543489,
543490,
543497,
543498,
543499,
543500,
543501,
543512,
543513,
543514,
543515,
543516,
543517,
543518,
543519,
543520,
543521,
543522,
543523,
543524,
543525,
543526,
543527,
543528,
543529,
543530,
543531,
543532,
543533,
543534,
543535,
543536,
543537,
543538,
543539,
543540,
543541,
543542,
543543,
543544,
543545,
543546,
543547,
543563,
543568,
543569,
543570,
543571,
543573,
543574,
543575,
543576,
543577,
543578,
543579,
543590,
543591,
543593,
543594,
543595,
543596,
543597,
543598,
543599,
543600,
543605,
543606,
543607,
543608,
543613,
543614,
543615,
543616,
543617,
543618,
543619,
543620,
543621,
543622,
543623,
543624,
543625,
543626,
543627,
543628,
543635,
543636,
543637,
543638,
543643,
543644,
543645,
543650,
543651,
543652,
543653,
543654,
543656,
543657,
543663,
543664,
543665,
543666,
543667,
543668,
543669,
543670,
543671,
543677,
543678,
543686,
543687,
543688,
543689,
543709,
543710,
543711,
543712,
543713,
543714,
543715,
543720,
543725,
543732,
543737,
543738,
543743,
543744,
543745,
543746,
543747,
543748,
543753,
543754,
543765,
543766,
543767,
543768,
543769,
543774,
543775,
543776,
543782,
543787,
543798,
543799,
543804,
543805,
543806,
543811,
543812,
543814,
543819,
543828,
543829,
543830,
543831,
543843,
543848,
543853,
543858,
543860,
543861,
543874,
543895,
543896,
543897,
543898,
543901,
543902,
543904,
543905,
543910,
543911,
543912,
543914,
543915,
543916,
543917,
543918,
543919,
543920,
543921,
543923,
543924,
543926,
543927,
543928,
543929,
543930,
543931,
543932,
543933,
543934,
543935,
543936,
543937,
543938,
543939,
543940,
543941,
543942,
543943,
543944,
543945,
543947,
543948,
543949,
543950,
543951,
543952,
543953,
543954,
543955,
543956,
543957,
543958,
543959,
543960,
543962,
543963,
543965,
543969,
543970,
543971,
543972,
543974,
543975,
543976,
543977,
543978,
543979,
543980,
543981,
543982,
543983,
543984,
543985,
543986,
543987,
543988,
543989,
543990,
543991,
543992,
543993,
543994,
543995,
543996,
543997,
543998,
543999,
544000,
544001,
544002,
544003,
544004,
544006,
544007,
544008,
544009,
544011,
544012,
544013,
544014,
544015,
544020,
544021,
544022,
544023,
544025,
544026,
544027,
544028,
544029,
544030,
544035,
544036,
544037,
544042,
544044,
544045,
544046,
544047,
544052,
544053,
544054,
544055,
544056,
544057,
544058,
544059,
544060,
544061,
544066,
544067,
544072,
544073,
544074,
544079,
544080,
544081,
544082,
544083,
544088,
544090,
544091,
544092,
544093,
544094,
544095,
544100,
544101,
544102,
544105,
544106,
544107,
544108,
544109,
544110,
544111,
544112,
544117,
544118,
544119,
544120,
544121,
544122,
544123,
544124,
544129,
544130,
544131,
544133,
544134,
544135,
544136,
544138,
544139,
544140,
544141,
544142,
544143,
544144,
544149,
544150,
544151,
544156,
544157,
544158,
544160,
544161,
544162,
544163,
544164,
544165,
544166,
544167,
544168,
544169,
544170,
544171,
555555,
570001,
570002,
570004,
570005,
590003,
590005,
590006,
590013,
590018,
590021,
590024,
590025,
590030,
590031,
590041,
590051,
590056,
590057,
590062,
590065,
590066,
590068,
590070,
590071,
590072,
590073,
590075,
590078,
590086,
590103,
590104,
590106,
590107,
590108,
590109,
590110,
590115,
590134,
590136,
590137,
590138,
750854,
750856,
750857,
750858,
750859,
780014,
780016,
780018,
100087,
100112,
100228,
100251,
100253,
100290,
100295,
100312,
100425,
100440,
100477,
100483,
100547,
100790,
100850,
117334,
126371,
132134,
132400,
132477,
132488,
132522,
132541,
132977,
140005,
533002,
533004,
533139,
533140,
533141,
533142,
533215,
533417,
533418,
533428,
533429,
533478,
533479,
533526,
533527,
535371,
535374,
535375,
535439,
535440,
535441,
536034,
536035,
536036,
536058,
536060,
537560,
537561,
537562,
537563,
537941,
537942,
537943,
537944,
538270,
538271,
538272,
538341,
538428,
538429,
538430,
538431,
538522,
538523,
538524,
538525,
539269,
539270,
539271,
539272,
540239,
540240,
540241,
540242,
541097,
541946,
542536,
542537,
542539,
542541,
542552,
542553,
542555,
542661,
542662,
542663,
542664,
542909,
543071,
543072,
543073,
543074,
543075,
543076,
543077,
543078,
543079,
543080,
543081,
543082,
543083,
543084,
543085,
543086,
543087,
543088,
543089,
543090,
543091,
543092,
543097,
543098,
543099,
543100,
543101,
543102,
543103,
543104,
543105,
543106,
543107,
543108,
543109,
543110,
543111,
543112,
543113,
543114,
543115,
543116,
543117,
543118,
543119,
543120,
543121,
543122,
543123,
543124,
543125,
543126,
543127,
543128,
543129,
543130,
543131,
543132,
543133,
543134,
543135,
543136,
543137,
543138,
543139,
543140,
543157,
543158,
543159,
543160,
543161,
543162,
543163,
543164,
543165,
543166,
543167,
543203,
543204,
543205,
543206,
543215,
543216,
543250,
543293,
543294,
543295,
543296,
543301,
543302,
543303,
543304,
543313,
543314,
543315,
543316,
543337,
543338,
543339,
543340,
543342,
543343,
543344,
543345,
543353,
543354,
543355,
543356,
543359,
543360,
543361,
543362,
543368,
543369,
543370,
543371,
543379,
543380,
543381,
543382,
543392,
543393,
543394,
543395,
543406,
543407,
543408,
543409,
543418,
543421,
543422,
543423,
543424,
543429,
543430,
543431,
543432,
543445,
543446,
543447,
543448,
543455,
543456,
543457,
543459,
543466,
543467,
543468,
543469,
543476,
543477,
543478,
543479,
543480,
543483,
543484,
543485,
543486,
543487,
543488,
543491,
543492,
543493,
543494,
543495,
543496,
543506,
543507,
543508,
543509,
543510,
543511,
543548,
543549,
543550,
543551,
543557,
543558,
543559,
543560,
543561,
543562,
543564,
543565,
543566,
543567,
543584,
543585,
543586,
543587,
543588,
543589,
543601,
543602,
543603,
543604,
543609,
543610,
543611,
543612,
543629,
543630,
543631,
543632,
543633,
543634,
543658,
543659,
543660,
543661,
543662,
543672,
543673,
543674,
543675,
543676,
543690,
543691,
543692,
543693,
543694,
543695,
543696,
543697,
543698,
543699,
543700,
543701,
543702,
543703,
543704,
543726,
543727,
543728,
543729,
543730,
543731,
543739,
543740,
543741,
543742,
543749,
543750,
543751,
543752,
543755,
543756,
543757,
543758,
543783,
543784,
543785,
543786,
543788,
543789,
543790,
543791,
543792,
543793,
543794,
543795,
543796,
543797,
543800,
543801,
543802,
543803,
543813,
543815,
543816,
543817,
543818,
543824,
543825,
543826,
543827,
543832,
543833,
543834,
543835,
543836,
543837,
543838,
543839,
543840,
543841,
543842,
543871,
543872,
543873,
543875,
543876,
543877,
543878,
543879,
543880,
543881,
543882,
543883,
543884,
543885,
543886,
543887,
543888,
543889,
543890,
543891,
543892,
543893,
543946,
543964,
543966,
543967,
543968,
543973,
544010,
544068,
544069,
544070,
544071,
544089,
544103,
544104,
544125,
544126,
544127,
544128,
544145,
544146,
544147,
544148,
544152,
544153,
544154,
544155,
544159,
590096,
540526,
540565,
541300,
542543,
542602,
543217,
543225,
543261,
543290,
543385,
543655,
543859,
543899,
543913,
543925,
544005,
544137,
929001,
929002,
929003,
929004,
929006,
929007,
929008,
929009,
929010,
929011,
929012,
929013,
929014,
929015,
929016,
929017,
929018,
929019,
929020,
929021,
929022,
958963,
973868,
700135,
710057,
715004,
715005,
715006,
715015,
715016,
715017,
715018,
715019,
715020,
715021,
715024,
717504,
912254,
912463,
935321,
935323,
935351,
935353,
935383,
935504,
935506,
935508,
935510,
935512,
935514,
935538,
935540,
935542,
935544,
935546,
935548,
935566,
935568,
935570,
935572,
935574,
935576,
935578,
935580,
935582,
935584,
935610,
935612,
935614,
935616,
935618,
935620,
935636,
935638,
935640,
935642,
935660,
935662,
935664,
935666,
935668,
935670,
935672,
935674,
935676,
935678,
935680,
935682,
935684,
935686,
935688,
935690,
935738,
935740,
935750,
935752,
935758,
935760,
935786,
935788,
935790,
935856,
935858,
935860,
935862,
935864,
935866,
935868,
935870,
935966,
936020,
936036,
936038,
936040,
936042,
936044,
936046,
936048,
936082,
936096,
936098,
936130,
936132,
936168,
936224,
936226,
936230,
936236,
936252,
936254,
936276,
936278,
936280,
936282,
936292,
936294,
936308,
936310,
936320,
936322,
936326,
936332,
936354,
936374,
936384,
936386,
936398,
936400,
936410,
936412,
936414,
936416,
936448,
936450,
936454,
936460,
936476,
936478,
936492,
936526,
936528,
936530,
936532,
936572,
936574,
936576,
936578,
936620,
936622,
936634,
936636,
936656,
936690,
936692,
936694,
936722,
936724,
936730,
936736,
936742,
936744,
936758,
936760,
936762,
936774,
936776,
936778,
936782,
936784,
936790,
936792,
936796,
936798,
936804,
936806,
936808,
936810,
936812,
936814,
936824,
936826,
936840,
936842,
936844,
936854,
936866,
936868,
936870,
936882,
936884,
936886,
936900,
936902,
936906,
936930,
936936,
936942,
936944,
936953,
936955,
936957,
936959,
936969,
936971,
936973,
936975,
936977,
936989,
936991,
936993,
937007,
937009,
937011,
937013,
937015,
937017,
937023,
937029,
937035,
937037,
937043,
937045,
937047,
937049,
937051,
937053,
937055,
937057,
937059,
937063,
937065,
937069,
937071,
937075,
937085,
937087,
937089,
937091,
937093,
937101,
937107,
937115,
937125,
937127,
937129,
937141,
937143,
937145,
937147,
937149,
937155,
937161,
937167,
937169,
937181,
937183,
937185,
937187,
937189,
937191,
937201,
937203,
937205,
937207,
937217,
937219,
937221,
937237,
937239,
937241,
937247,
937253,
937259,
937269,
937271,
937273,
937285,
937287,
937289,
937295,
937301,
937307,
937311,
937315,
937319,
937329,
937331,
937333,
937339,
937341,
937343,
937345,
937347,
937351,
937355,
937359,
937363,
937365,
937367,
937369,
937371,
937373,
937375,
937377,
937379,
937381,
937383,
937385,
937395,
937397,
937399,
937401,
937407,
937409,
937411,
937417,
937419,
937427,
937429,
937431,
937433,
937439,
937441,
937443,
937449,
937451,
937453,
937455,
937457,
937459,
937461,
937465,
937467,
937469,
937471,
937473,
937483,
937485,
937487,
937489,
937495,
937497,
937499,
937501,
937503,
937511,
937513,
937515,
937517,
937523,
937525,
937527,
937529,
937531,
937533,
937535,
937537,
937539,
937541,
937545,
937547,
937549,
937551,
937555,
937557,
937559,
937561,
937571,
937573,
937575,
937577,
937579,
937581,
937587,
937589,
937591,
937593,
937595,
937597,
937609,
937611,
937613,
937615,
937617,
937619,
937621,
937623,
937625,
937627,
937629,
937631,
937633,
937635,
937637,
937639,
937647,
937649,
937651,
937653,
937667,
937669,
937671,
937673,
937675,
937677,
937679,
937681,
937683,
937685,
937687,
937693,
937695,
937697,
937699,
937701,
937703,
937705,
937707,
937709,
937711,
937717,
937719,
937721,
937723,
937725,
937735,
937737,
937739,
937741,
937743,
937745,
937747,
937759,
937761,
937763,
937765,
937767,
937771,
937773,
937775,
937777,
937781,
937783,
937785,
937793,
937795,
937797,
937799,
937801,
937803,
937805,
937807,
937815,
937817,
937819,
937821,
937823,
937825,
937827,
937829,
937831,
937833,
937835,
937849,
937851,
937853,
937855,
937857,
937859,
937861,
937863,
937865,
937875,
937877,
937879,
937883,
937885,
937887,
937889,
937893,
937895,
937897,
937899,
937917,
937919,
937921,
937923,
937925,
937929,
937931,
937933,
937935,
937937,
937939,
937941,
937955,
937957,
937959,
937961,
937963,
937965,
937967,
937969,
937971,
937977,
937979,
937981,
937983,
937985,
937987,
937993,
937995,
937997,
937999,
938001,
938003,
938005,
938007,
938009,
938011,
938013,
938015,
938019,
938021,
938023,
938025,
938027,
938029,
938039,
938041,
938043,
938051,
938053,
938055,
938061,
938063,
938065,
938067,
938069,
938071,
938073,
938075,
938077,
938087,
938089,
938091,
938093,
938095,
938097,
938099,
938101,
938103,
938105,
938107,
938109,
938111,
938113,
938117,
938119,
938121,
938123,
938125,
938127,
938129,
938131,
938133,
938135,
938137,
938139,
938141,
938143,
938147,
938149,
938151,
938154,
938156,
938158,
938160,
938162,
938164,
938166,
938168,
938170,
938172,
938174,
938176,
938178,
938180,
938188,
938190,
938192,
938194,
938196,
938198,
938200,
938202,
938204,
938206,
938208,
938210,
938212,
938214,
938216,
938218,
938220,
938222,
938224,
938226,
938228,
938230,
938232,
938234,
938236,
938238,
938240,
938242,
938244,
938246,
938248,
938250,
938252,
938254,
938256,
938258,
938260,
938262,
938264,
938266,
938268,
938270,
938272,
938274,
938276,
938278,
938280,
938282,
938284,
938286,
938288,
938290,
938292,
938294,
938296,
938298,
938300,
938302,
938304,
938306,
938308,
938310,
938312,
938314,
938316,
938318,
938320,
938324,
938326,
938328,
938330,
938332,
938334,
938336,
938338,
938340,
938342,
938344,
938346,
938348,
938350,
938352,
938354,
938356,
938358,
938360,
938362,
938364,
938366,
938368,
938370,
938372,
938374,
938376,
938378,
938380,
938382,
938384,
938386,
938388,
938390,
938392,
938394,
938396,
938398,
938400,
938402,
938404,
938406,
938408,
938410,
938412,
938414,
938416,
938418,
938420,
938422,
938424,
938426,
938428,
938430,
938432,
938434,
938436,
938438,
938440,
938442,
938444,
938446,
938450,
938452,
938454,
938456,
938458,
938460,
938462,
938464,
938466,
938468,
938470,
938472,
938474,
938476,
938478,
938480,
938482,
938484,
938486,
938488,
938490,
938492,
938494,
938496,
938498,
938500,
938502,
938504,
938506,
938508,
938510,
938512,
938514,
938516,
938518,
938520,
938522,
938524,
938526,
938528,
938530,
938532,
938534,
938536,
938538,
938540,
938542,
938544,
938546,
938548,
938550,
938552,
938554,
938556,
938558,
938560,
938562,
938564,
938566,
938568,
938570,
938572,
938574,
938576,
938578,
938580,
938582,
938584,
938586,
938588,
938590,
938592,
938594,
938596,
938598,
938600,
938602,
938604,
938606,
938608,
938610,
938612,
938614,
938616,
938618,
938620,
938622,
938624,
938626,
938628,
938630,
938632,
938634,
938636,
938638,
938640,
938642,
938644,
938646,
938648,
938650,
938652,
938654,
938656,
938658,
938660,
938662,
938664,
938666,
938668,
938670,
938672,
938674,
938676,
938678,
938680,
938682,
938684,
938686,
938688,
938690,
938692,
938694,
938696,
938698,
938700,
938702,
938704,
938706,
938708,
938710,
938712,
938714,
938716,
938718,
938720,
938722,
938724,
938726,
938728,
938730,
938732,
938734,
938736,
938738,
938740,
938742,
938744,
938746,
938748,
938750,
938752,
938754,
938756,
938758,
938760,
938762,
938764,
938766,
938768,
938770,
938772,
938774,
938776,
938778,
938780,
938782,
938784,
938786,
938788,
938790,
938792,
938794,
938796,
938798,
938800,
938802,
938804,
938806,
938808,
938810,
938812,
938814,
938816,
938818,
938820,
938822,
938824,
938826,
938828,
938830,
938832,
938834,
938836,
938838,
938840,
938842,
938844,
938846,
938848,
938850,
938852,
938854,
938856,
938858,
938860,
938862,
938864,
938866,
938868,
938870,
938872,
938874,
938876,
938878,
938880,
938882,
938884,
938886,
938888,
938890,
938892,
938894,
938896,
938898,
938900,
938902,
938904,
938906,
938908,
938910,
938912,
938914,
938916,
938918,
938920,
938922,
938924,
938926,
938928,
938930,
938932,
938934,
938936,
938938,
938940,
938942,
938944,
938946,
938948,
938952,
938954,
938956,
938958,
938960,
938962,
938964,
938966,
938968,
938970,
938972,
938974,
938976,
938978,
938980,
938982,
938984,
938986,
938988,
938990,
938992,
938994,
938996,
938998,
939000,
939002,
939004,
939006,
939008,
939010,
939012,
939014,
939016,
939018,
939020,
939022,
939024,
939026,
939028,
939030,
939032,
939034,
939036,
939038,
939040,
939042,
939044,
939046,
939048,
939050,
939052,
939054,
939056,
939058,
939060,
939062,
939064,
939066,
939068,
939070,
939072,
939074,
939076,
939078,
939080,
939082,
939084,
939086,
939088,
939090,
939092,
939094,
939096,
939098,
939100,
939102,
939104,
939106,
939108,
939110,
939112,
939114,
939116,
939118,
939120,
939122,
939124,
939126,
939128,
939130,
939132,
939134,
939136,
939138,
939140,
939142,
939144,
939146,
939148,
939150,
939152,
939154,
939156,
939158,
939160,
939162,
939164,
939166,
939168,
939170,
939172,
939174,
939176,
939178,
939180,
939182,
939184,
939186,
939188,
939190,
939192,
939194,
939196,
939198,
939200,
939202,
939204,
939206,
939208,
939210,
939212,
939214,
939216,
939218,
939220,
939222,
939224,
939226,
939228,
939230,
939232,
939234,
939236,
939238,
939240,
939242,
939244,
939246,
939248,
939250,
939252,
939254,
939256,
939258,
939260,
939262,
939264,
939266,
939268,
939270,
939272,
939274,
939276,
939278,
939280,
939282,
939284,
939286,
939288,
939290,
939293,
939295,
939297,
939299,
939301,
939303,
939305,
939307,
939309,
939311,
939313,
939315,
939317,
939319,
939321,
939323,
939325,
939327,
939329,
939331,
939333,
939335,
939337,
939339,
939341,
939343,
939345,
939347,
939349,
939351,
939353,
939355,
939357,
939359,
939361,
939363,
939365,
939367,
939369,
939371,
939373,
939375,
939377,
939379,
939381,
939383,
939385,
939387,
939389,
939391,
939393,
939395,
939397,
939399,
939401,
939403,
939405,
939407,
939409,
939411,
939413,
939415,
939417,
939419,
939421,
939423,
939425,
939427,
939429,
939431,
939433,
939435,
939437,
939439,
939441,
939443,
939445,
939447,
939449,
939451,
939453,
939455,
939457,
939459,
939461,
939463,
939465,
939467,
946764,
946831,
946997,
946998,
947001,
947841,
948103,
948802,
949156,
949250,
949342,
949439,
949450,
949451,
949470,
949491,
949563,
949719,
950062,
950199,
950201,
950202,
950203,
950204,
950205,
950206,
950207,
950208,
950210,
950211,
950212,
950213,
950214,
950215,
950216,
950217,
950218,
950219,
950220,
950221,
950306,
950322,
950360,
950372,
950373,
950380,
950434,
950446,
950457,
950458,
950459,
950460,
950461,
950462,
950463,
950464,
950465,
950467,
950468,
950469,
950471,
950473,
950475,
950485,
950486,
950487,
950492,
950494,
950497,
950498,
950600,
950619,
950641,
950668,
950675,
950701,
950702,
950707,
950735,
950736,
950737,
950738,
950742,
950750,
950751,
950752,
950753,
950754,
950762,
950763,
950778,
950803,
950804,
950860,
950868,
950871,
950898,
950924,
950931,
951024,
951043,
951044,
951047,
951064,
951069,
951096,
951099,
951115,
951143,
951170,
951206,
951208,
951209,
951210,
951211,
951212,
951213,
951214,
951217,
951242,
951244,
951246,
951248,
951249,
951270,
951276,
951277,
951279,
951289,
951304,
951306,
951325,
951336,
951355,
951366,
951381,
951398,
951410,
951412,
951431,
951482,
951486,
951502,
951507,
951511,
951512,
951520,
951521,
951522,
951523,
951524,
951525,
951537,
951556,
951563,
951589,
951594,
951597,
951599,
951600,
951612,
951623,
951634,
951646,
951647,
951650,
951651,
951653,
951654,
951677,
951678,
951679,
951680,
951688,
951689,
951695,
951709,
951710,
951713,
951716,
951718,
951720,
951729,
951756,
951778,
951799,
951822,
951847,
951853,
951856,
951869,
951882,
951890,
951898,
951899,
951927,
951931,
951949,
951967,
951971,
951995,
952007,
952011,
952019,
952030,
952034,
952050,
952056,
952077,
952095,
952105,
952136,
952140,
952171,
952210,
952219,
952249,
952267,
952268,
952288,
952296,
952316,
952322,
952324,
952360,
952361,
952362,
952364,
952414,
952426,
952508,
952525,
952545,
952546,
952547,
952548,
952549,
952550,
952566,
952567,
952574,
952579,
952600,
952610,
952625,
952636,
952658,
952659,
952661,
952662,
952676,
952680,
952728,
952762,
952774,
952778,
952789,
952791,
952793,
952810,
952829,
952832,
952834,
952847,
952850,
952851,
952855,
952863,
952884,
952885,
952894,
952901,
952906,
952914,
952917,
952920,
952960,
952965,
952984,
952985,
952989,
952990,
952993,
953010,
953059,
953060,
953061,
953062,
953063,
953064,
953065,
953066,
953067,
953077,
953095,
953102,
953107,
953113,
953139,
953144,
953148,
953153,
953154,
953176,
953177,
953179,
953180,
953197,
953220,
953223,
953224,
953225,
953235,
953237,
953250,
953254,
953262,
953266,
953271,
953272,
953276,
953285,
953289,
953383,
953385,
953401,
953403,
953413,
953414,
953417,
953420,
953427,
953428,
953431,
953432,
953433,
953434,
953435,
953436,
953437,
953438,
953439,
953440,
953441,
953449,
953453,
953460,
953489,
953490,
953497,
953498,
953499,
953501,
953511,
953518,
953519,
953520,
953521,
953522,
953550,
953585,
953589,
953618,
953621,
953649,
953659,
953670,
953674,
953675,
953677,
953682,
953685,
953687,
953688,
953700,
953711,
953726,
953739,
953740,
953751,
953755,
953756,
953764,
953792,
953806,
953833,
953876,
953877,
953885,
953888,
953894,
953896,
953899,
953904,
953906,
953907,
953943,
953948,
953950,
953957,
953965,
953986,
953988,
954000,
954005,
954028,
954029,
954040,
954041,
954048,
954052,
954056,
954058,
954076,
954113,
954131,
954142,
954166,
954171,
954176,
954183,
954185,
954226,
954231,
954232,
954264,
954287,
954299,
954307,
954310,
954311,
954318,
954362,
954364,
954375,
954378,
954382,
954394,
954408,
954427,
954435,
954442,
954448,
954461,
954464,
954481,
954508,
954527,
954533,
954534,
954549,
954576,
954588,
954598,
954614,
954622,
954634,
954648,
954665,
954678,
954696,
954704,
954707,
954708,
954709,
954711,
954712,
954721,
954722,
954723,
954724,
954725,
954726,
954728,
954730,
954731,
954734,
954738,
954741,
954744,
954781,
954795,
954796,
954797,
954798,
954799,
954815,
954841,
954850,
954853,
954855,
954872,
954875,
954921,
954922,
954925,
954934,
954935,
954975,
954977,
954979,
954984,
954989,
954991,
955000,
955027,
955031,
955032,
955052,
955054,
955087,
955089,
955091,
955092,
955094,
955113,
955126,
955145,
955147,
955156,
955174,
955183,
955207,
955208,
955229,
955248,
955251,
955256,
955257,
955282,
955294,
955310,
955311,
955319,
955329,
955331,
955333,
955359,
955389,
955398,
955431,
955442,
955443,
955445,
955462,
955483,
955484,
955488,
955498,
955512,
955513,
955522,
955542,
955561,
955584,
955588,
955590,
955600,
955603,
955610,
955625,
955641,
955693,
955697,
955770,
955771,
955772,
955782,
955784,
955805,
955815,
955817,
955840,
955845,
955852,
955858,
955882,
955902,
955912,
955942,
955958,
955986,
955992,
955999,
956041,
956046,
956048,
956063,
956064,
956065,
956066,
956067,
956068,
956069,
956070,
956071,
956072,
956073,
956074,
956075,
956076,
956077,
956078,
956092,
956100,
956136,
956141,
956143,
956148,
956149,
956150,
956177,
956189,
956193,
956208,
956233,
956235,
956260,
956276,
956311,
956318,
956321,
956322,
956337,
956338,
956426,
956427,
956470,
956508,
956528,
956530,
956532,
956533,
956535,
956536,
956540,
956552,
956563,
956577,
956582,
956585,
956588,
956594,
956597,
956607,
956617,
956618,
956620,
956631,
956634,
956636,
956655,
956666,
956670,
956683,
956692,
956700,
956707,
956710,
956714,
956717,
956718,
956732,
956752,
956760,
956772,
956774,
956775,
956787,
956797,
956799,
956802,
956808,
956812,
956820,
956821,
956835,
956836,
956844,
956850,
956863,
956874,
956881,
956882,
956883,
956889,
956892,
956893,
956902,
956903,
956904,
956905,
956909,
956919,
956922,
956923,
956927,
956929,
956930,
956936,
956939,
956948,
956955,
957039,
957050,
957052,
957067,
957073,
957077,
957083,
957090,
957093,
957102,
957109,
957121,
957129,
957130,
957131,
957132,
957133,
957134,
957135,
957144,
957166,
957170,
957172,
957173,
957179,
957180,
957186,
957187,
957189,
957192,
957196,
957197,
957206,
957207,
957208,
957209,
957220,
957222,
957223,
957225,
957228,
957232,
957233,
957234,
957235,
957236,
957237,
957246,
957253,
957280,
957360,
957375,
957381,
957385,
957393,
957398,
957402,
957409,
957419,
957422,
957441,
957442,
957443,
957444,
957445,
957446,
957463,
957468,
957481,
957530,
957531,
957542,
957543,
957549,
957615,
957633,
957634,
957635,
957636,
957637,
957638,
957639,
957640,
957641,
957642,
957643,
957644,
957645,
957646,
957647,
957649,
957656,
957670,
957671,
957680,
957683,
957690,
957691,
957704,
957707,
957712,
957713,
957725,
957727,
957744,
957746,
957761,
957762,
957764,
957765,
957770,
957788,
957790,
957799,
957801,
957807,
957808,
957809,
957810,
957824,
957833,
957835,
957836,
957837,
957844,
957855,
957863,
957864,
957865,
957874,
957877,
957889,
957898,
957905,
957907,
957915,
957921,
957924,
957932,
957951,
957953,
957962,
957969,
957970,
957977,
957983,
957988,
957989,
957996,
957998,
958011,
958015,
958016,
958017,
958020,
958024,
958025,
958047,
958048,
958065,
958067,
958101,
958116,
958127,
958138,
958156,
958165,
958166,
958171,
958172,
958176,
958177,
958178,
958179,
958180,
958190,
958191,
958192,
958213,
958216,
958219,
958226,
958232,
958245,
958246,
958257,
958258,
958259,
958275,
958278,
958280,
958281,
958287,
958288,
958291,
958300,
958304,
958306,
958307,
958311,
958312,
958313,
958332,
958338,
958349,
958353,
958354,
958356,
958364,
958385,
958393,
958394,
958397,
958399,
958400,
958406,
958410,
958411,
958418,
958419,
958433,
958438,
958445,
958447,
958463,
958464,
958466,
958467,
958468,
958469,
958487,
958489,
958502,
958511,
958514,
958518,
958522,
958523,
958531,
958538,
958539,
958540,
958542,
958551,
958553,
958555,
958565,
958568,
958569,
958575,
958583,
958587,
958588,
958596,
958599,
958607,
958609,
958610,
958612,
958613,
958614,
958616,
958618,
958625,
958629,
958641,
958646,
958647,
958655,
958659,
958661,
958662,
958664,
958668,
958670,
958673,
958674,
958675,
958680,
958683,
958685,
958687,
958688,
958692,
958696,
958737,
958738,
958744,
958747,
958753,
958759,
958764,
958772,
958773,
958779,
958788,
958789,
958795,
958801,
958805,
958806,
958807,
958808,
958811,
958814,
958818,
958821,
958825,
958828,
958831,
958834,
958838,
958840,
958842,
958845,
958846,
958851,
958853,
958855,
958857,
958866,
958869,
958874,
958875,
958877,
958878,
958879,
958884,
958887,
958889,
958891,
958892,
958893,
958895,
958896,
958897,
958901,
958909,
958911,
958912,
958913,
958915,
958916,
958922,
958928,
958930,
958934,
958936,
958938,
958943,
958944,
958945,
958954,
958955,
958959,
958965,
958968,
958972,
958977,
958978,
958980,
958981,
958982,
958989,
958995,
959000,
959008,
959009,
959012,
959013,
959025,
959027,
959031,
959033,
959034,
959036,
959041,
959042,
959043,
959045,
959052,
959056,
959058,
959059,
959060,
959066,
959068,
959072,
959078,
959079,
959082,
959085,
959086,
959087,
959092,
959094,
959096,
959103,
959105,
959112,
959117,
959118,
959119,
959122,
959126,
959134,
959138,
959144,
959149,
959151,
959152,
959153,
959155,
959159,
959160,
959162,
959181,
959182,
959183,
959184,
959185,
959186,
959187,
959190,
959191,
959192,
959193,
959194,
959196,
959198,
959205,
959206,
959208,
959209,
959210,
959211,
959213,
959215,
959216,
959219,
959224,
959225,
959227,
959228,
959229,
959230,
959231,
959233,
959237,
959242,
959244,
959245,
959247,
959250,
959256,
959275,
959282,
959284,
959285,
959286,
959287,
959288,
959289,
959290,
959296,
959297,
959298,
959300,
959307,
959308,
959311,
959312,
959313,
959314,
959315,
959316,
959320,
959321,
959324,
959325,
959337,
959338,
959339,
959345,
959348,
959352,
959353,
959354,
959355,
959369,
959374,
959378,
959379,
959380,
959381,
959382,
959386,
959389,
959397,
959398,
959401,
959411,
959412,
959423,
959428,
959431,
959432,
959435,
959442,
959449,
959454,
959456,
959457,
959458,
959459,
959460,
959461,
959473,
959478,
959484,
959489,
959493,
959495,
959496,
959501,
959502,
959507,
959508,
959515,
959516,
959537,
959542,
959550,
959552,
959566,
959570,
959572,
959581,
959620,
959637,
959644,
959646,
959666,
959681,
959690,
959692,
959701,
959712,
959721,
959728,
959735,
959750,
959754,
959756,
959760,
959762,
959763,
959774,
959780,
959790,
959792,
959794,
959795,
959799,
959800,
959806,
959818,
959820,
959823,
959825,
959827,
959829,
959831,
959834,
959838,
959839,
959844,
959863,
959867,
959872,
959881,
959887,
959888,
959891,
959894,
959895,
959916,
959942,
959943,
959947,
959957,
959974,
959978,
959993,
960004,
960026,
960031,
960033,
960037,
960039,
960048,
960065,
960066,
960070,
960072,
960073,
960075,
960076,
960082,
960111,
960116,
960117,
960121,
960125,
960130,
960134,
960136,
960145,
960148,
960149,
960150,
960151,
960152,
960153,
960159,
960160,
960161,
960167,
960168,
960169,
960174,
960175,
960177,
960179,
960183,
960185,
960190,
960191,
960203,
960207,
960212,
960217,
960221,
960225,
960226,
960229,
960232,
960238,
960239,
960240,
960241,
960242,
960243,
960244,
960248,
960250,
960255,
960256,
960265,
960267,
960281,
960282,
960297,
960301,
960309,
960312,
960313,
960319,
960323,
960336,
960337,
960338,
960346,
960349,
960353,
960354,
960355,
960358,
960361,
960362,
960364,
960366,
960371,
960372,
960379,
960381,
960382,
960390,
960399,
960403,
960405,
960409,
960410,
960412,
960413,
960416,
960418,
960419,
960420,
960422,
960424,
960426,
960428,
960429,
960430,
960431,
960435,
960436,
960437,
960438,
960441,
960448,
960449,
960454,
960461,
960462,
960463,
960467,
960469,
960472,
960473,
960476,
960477,
960478,
960479,
960480,
960481,
960482,
960483,
960484,
960485,
960486,
960487,
960489,
960491,
960492,
960495,
961400,
961403,
961707,
961708,
961713,
961714,
961717,
961718,
961728,
961730,
961732,
961734,
961744,
961749,
961751,
961753,
961754,
961756,
961758,
961760,
961763,
961765,
961767,
961770,
961771,
961773,
961776,
961777,
961779,
961780,
961782,
961783,
961785,
961786,
961788,
961789,
961791,
961792,
961795,
961796,
961797,
961798,
961800,
961801,
961803,
961804,
961806,
961807,
961809,
961810,
961812,
961813,
961815,
961816,
961818,
961819,
961821,
961822,
961825,
961826,
961828,
961830,
961833,
961835,
961839,
961841,
961845,
961847,
961851,
961853,
961857,
961859,
961863,
961865,
961869,
961871,
961875,
961877,
961881,
961885,
961889,
961891,
961895,
961897,
961900,
961902,
961904,
961906,
961908,
961910,
961917,
961919,
972365,
972366,
972367,
972368,
972459,
972471,
972475,
972478,
972479,
972496,
972507,
972513,
972523,
972531,
972551,
972556,
972557,
972558,
972562,
972563,
972571,
972574,
972575,
972576,
972577,
972578,
972579,
972580,
972581,
972582,
972583,
972584,
972585,
972591,
972599,
972607,
972649,
972650,
972655,
972657,
972687,
972691,
972693,
972706,
972711,
972715,
972716,
972721,
972722,
972730,
972731,
972745,
972746,
972763,
972768,
972771,
972772,
972773,
972778,
972779,
972781,
972788,
972790,
972791,
972801,
972805,
972821,
972826,
972832,
972848,
972857,
972859,
972862,
972867,
972868,
972873,
972874,
972878,
973001,
973003,
973007,
973008,
973014,
973018,
973019,
973027,
973036,
973037,
973038,
973045,
973047,
973048,
973051,
973052,
973053,
973073,
973078,
973079,
973085,
973086,
973087,
973092,
973098,
973103,
973106,
973111,
973116,
973118,
973124,
973125,
973126,
973127,
973143,
973150,
973152,
973153,
973154,
973159,
973163,
973171,
973175,
973176,
973181,
973185,
973188,
973190,
973201,
973204,
973209,
973211,
973212,
973213,
973215,
973216,
973219,
973220,
973221,
973222,
973223,
973225,
973228,
973229,
973230,
973232,
973235,
973236,
973237,
973239,
973240,
973249,
973250,
973253,
973259,
973260,
973261,
973262,
973265,
973269,
973271,
973272,
973273,
973274,
973275,
973276,
973277,
973278,
973282,
973283,
973284,
973285,
973286,
973287,
973288,
973289,
973290,
973291,
973292,
973293,
973294,
973295,
973301,
973302,
973303,
973304,
973308,
973309,
973314,
973315,
973320,
973321,
973324,
973325,
973326,
973327,
973330,
973332,
973335,
973337,
973338,
973339,
973340,
973341,
973344,
973345,
973348,
973350,
973352,
973361,
973362,
973363,
973365,
973366,
973367,
973369,
973372,
973374,
973377,
973380,
973383,
973385,
973386,
973391,
973395,
973396,
973400,
973404,
973405,
973407,
973408,
973409,
973410,
973411,
973412,
973414,
973419,
973420,
973421,
973422,
973424,
973434,
973435,
973436,
973438,
973439,
973440,
973442,
973443,
973444,
973445,
973450,
973452,
973453,
973459,
973460,
973463,
973469,
973470,
973471,
973472,
973474,
973475,
973476,
973484,
973485,
973486,
973489,
973491,
973496,
973498,
973499,
973500,
973501,
973504,
973505,
973508,
973509,
973510,
973511,
973512,
973513,
973518,
973519,
973520,
973521,
973522,
973524,
973525,
973537,
973539,
973540,
973541,
973542,
973543,
973544,
973545,
973546,
973547,
973548,
973550,
973553,
973554,
973556,
973557,
973559,
973560,
973562,
973564,
973567,
973568,
973570,
973572,
973573,
973574,
973575,
973579,
973581,
973582,
973583,
973584,
973585,
973586,
973587,
973589,
973592,
973594,
973595,
973596,
973597,
973602,
973603,
973604,
973606,
973608,
973609,
973610,
973611,
973616,
973618,
973623,
973625,
973626,
973627,
973629,
973630,
973632,
973635,
973638,
973642,
973645,
973646,
973647,
973650,
973652,
973654,
973655,
973659,
973661,
973662,
973663,
973664,
973665,
973666,
973667,
973668,
973669,
973671,
973672,
973673,
973674,
973675,
973678,
973679,
973680,
973681,
973682,
973684,
973685,
973687,
973688,
973689,
973690,
973691,
973692,
973693,
973694,
973695,
973697,
973698,
973699,
973701,
973703,
973704,
973705,
973706,
973710,
973711,
973712,
973715,
973716,
973717,
973718,
973720,
973721,
973722,
973725,
973726,
973727,
973728,
973729,
973730,
973731,
973733,
973734,
973735,
973736,
973737,
973738,
973741,
973742,
973743,
973744,
973746,
973748,
973749,
973750,
973751,
973752,
973753,
973754,
973755,
973756,
973757,
973758,
973759,
973760,
973761,
973762,
973763,
973764,
973766,
973775,
973776,
973777,
973779,
973783,
973784,
973785,
973786,
973787,
973788,
973790,
973792,
973794,
973796,
973798,
973799,
973800,
973802,
973803,
973805,
973806,
973807,
973811,
973813,
973814,
973816,
973817,
973818,
973820,
973822,
973823,
973824,
973825,
973827,
973828,
973829,
973830,
973831,
973832,
973833,
973834,
973836,
973837,
973841,
973842,
973844,
973848,
973849,
973852,
973856,
973858,
973859,
973860,
973861,
973862,
973863,
973864,
973867,
973869,
973870,
973871,
973873,
973874,
973875,
973876,
973877,
973878,
973879,
973880,
973881,
973882,
973883,
973887,
973888,
973889,
973892,
973893,
973897,
973898,
973901,
973902,
973903,
973904,
973907,
973910,
973911,
973912,
973920,
973921,
973923,
973926,
973927,
973928,
973929,
973930,
973933,
973934,
973935,
973936,
973938,
973939,
973940,
973941,
973942,
973944,
973945,
973949,
973952,
973954,
973960,
973962,
973963,
973965,
973966,
973969,
973970,
973971,
973972,
973973,
973975,
973976,
973978,
973979,
973981,
973982,
973984,
973985,
973986,
973987,
973988,
973990,
973991,
973993,
973994,
973995,
973996,
973998,
973999,
974000,
974002,
974004,
974005,
974008,
974009,
974011,
974013,
974014,
974015,
974016,
974017,
974018,
974019,
974021,
974022,
974025,
974026,
974028,
974029,
974031,
974032,
974033,
974034,
974035,
974036,
974037,
974038,
974039,
974040,
974041,
974042,
974043,
974044,
974045,
974046,
974047,
974048,
974049,
974050,
974051,
974052,
974053,
974056,
974057,
974058,
974060,
974061,
974063,
974064,
974065,
974067,
974070,
974071,
974072,
974073,
974074,
974075,
974078,
974079,
974081,
974082,
974085,
974087,
974088,
974089,
974090,
974091,
974092,
974093,
974094,
974095,
974096,
974098,
974099,
974100,
974101,
974102,
974103,
974104,
974106,
974107,
974108,
974109,
974111,
974112,
974113,
974114,
974115,
974117,
974118,
974119,
974120,
974121,
974122,
974123,
974124,
974126,
974127,
974128,
974129,
974130,
974131,
974132,
974133,
974134,
974135,
974136,
974137,
974138,
974139,
974140,
974141,
974142,
974143,
974144,
974145,
974146,
974150,
974151,
974152,
974154,
974155,
974156,
974160,
974162,
974163,
974164,
974165,
974166,
974167,
974168,
974169,
974170,
974171,
974173,
974174,
974175,
974177,
974178,
974179,
974180,
974181,
974183,
974184,
974185,
974186,
974187,
974188,
974189,
974190,
974194,
974195,
974196,
974197,
974198,
974199,
974200,
974201,
974202,
974204,
974206,
974207,
974208,
974209,
974210,
974212,
974213,
974214,
974215,
974217,
974218,
974219,
974220,
974221,
974222,
974223,
974224,
974227,
974228,
974229,
974230,
974231,
974232,
974233,
974234,
974235,
974236,
974237,
974238,
974240,
974241,
974242,
974243,
974244,
974245,
974246,
974247,
974248,
974249,
974250,
974252,
974253,
974255,
974256,
974257,
974258,
974259,
974260,
974262,
974263,
974264,
974265,
974266,
974267,
974268,
974271,
974272,
974273,
974274,
974275,
974276,
974278,
974279,
974280,
974281,
974282,
974283,
974284,
974285,
974286,
974287,
974288,
974289,
974290,
974292,
974293,
974295,
974296,
974297,
974298,
974299,
974300,
974301,
974302,
974303,
974304,
974305,
974306,
974307,
974308,
974309,
974310,
974311,
974312,
974313,
974314,
974315,
974316,
974317,
974318,
974319,
974320,
974321,
974322,
974323,
974324,
974325,
974326,
974327,
974328,
974330,
974331,
974334,
974335,
974336,
974337,
974338,
974339,
974340,
974341,
974342,
974343,
974344,
974345,
974347,
974348,
974349,
974350,
974351,
974352,
974353,
974354,
974355,
974356,
974357,
974358,
974359,
974360,
974362,
974363,
974364,
974365,
974366,
974367,
974368,
974369,
974370,
974371,
974372,
974373,
974374,
974375,
974376,
974377,
974378,
974379,
974380,
974381,
974382,
974383,
974384,
974385,
974386,
974387,
974388,
974389,
974390,
974391,
974392,
974393,
974394,
974395,
974396,
974397,
974398,
974399,
974400,
974401,
974402,
974403,
974404,
974405,
974406,
974407,
974408,
974410,
974411,
974412,
974413,
974414,
974415,
974416,
974417,
974419,
974420,
974421,
974422,
974423,
974424,
974426,
974427,
974428,
974429,
974430,
974431,
974432,
974433,
974434,
974435,
974437,
974438,
974439,
974441,
974442,
974443,
974445,
974446,
974447,
974448,
974449,
974450,
974451,
974452,
974453,
974454,
974455,
974456,
974457,
974458,
974459,
974460,
974461,
974462,
974463,
974464,
974465,
974466,
974467,
974468,
974469,
974470,
974471,
974472,
974473,
974474,
974476,
974478,
974479,
974480,
974482,
974483,
974484,
974485,
974486,
974488,
974489,
974490,
974492,
974493,
974495,
974496,
974497,
974498,
974499,
974502,
974503,
974506,
974507,
974508,
974509,
974510,
974511,
974512,
974513,
974514,
974515,
974516,
974519,
974520,
974521,
974524,
974525,
974526,
974527,
974528,
974529,
974530,
974531,
974532,
974533,
974534,
974535,
974536,
974537,
974538,
974539,
974546,
974547,
974548,
974549,
974550,
974552,
974553,
974554,
974555,
974556,
974557,
974559,
974560,
974561,
974562,
974564,
974565,
974566,
974567,
974568,
974569,
974570,
974571,
974572,
974573,
974574,
974575,
974576,
974577,
974578,
974580,
974581,
974582,
974583,
974584,
974585,
974586,
974587,
974588,
974589,
974590,
974591,
974592,
974593,
974594,
974595,
974596,
974597,
974598,
974599,
974600,
974601,
974602,
974603,
974604,
974605,
974606,
974607,
974608,
974609,
974610,
974611,
974612,
974613,
974614,
974615,
974616,
974617,
974618,
974619,
974620,
974621,
974622,
974623,
974624,
974625,
974626,
974627,
974628,
974629,
974630,
974631,
974633,
974634,
974635,
974636,
974637,
974638,
974639,
974640,
974641,
974642,
974643,
974644,
974645,
974646,
974647,
974648,
974649,
974650,
974651,
974652,
974653,
974654,
974655,
974656,
974657,
974658,
974659,
974660,
974661,
974662,
974663,
974664,
974665,
974666,
974667,
974668,
974669,
974670,
974671,
974672,
974673,
974674,
974675,
974676,
974677,
974678,
974679,
974681,
974682,
974683,
974684,
974685,
974687,
974688,
974689,
974690,
974691,
974692,
974693,
974694,
974695,
974696,
974697,
974698,
974700,
974701,
974702,
974703,
974704,
974705,
974706,
974708,
974709,
974710,
974711,
974712,
974713,
974714,
974715,
974716,
974717,
974718,
974719,
974720,
974721,
974722,
974723,
974724,
974725,
974726,
974727,
974728,
974729,
974730,
974731,
974732,
974733,
974735,
974736,
974737,
974738,
974740,
974741,
974742,
974743,
974744,
974745,
974746,
974747,
974748,
974749,
974750,
974751,
974752,
974753,
974754,
974756,
974757,
974758,
974759,
974760,
974761,
974762,
974763,
974764,
974765,
974766,
974767,
974768,
974769,
974770,
974771,
974772,
974773,
974774,
974775,
974776,
974777,
974778,
974779,
974780,
974781,
974783,
974785,
974786,
974788,
974789,
974790,
974791,
974792,
974793,
974794,
974795,
974796,
974797,
974798,
974799,
974800,
974801,
974802,
974803,
974804,
974805,
974806,
974807,
974808,
974809,
974810,
974811,
974812,
974813,
974814,
974815,
974818,
974819,
974820,
974821,
974822,
974823,
974824,
974825,
974826,
974827,
974828,
974829,
974830,
974831,
974832,
974833,
974834,
974835,
974836,
974837,
974838,
974839,
974840,
974841,
974842,
974843,
974844,
974845,
974846,
974847,
974848,
974849,
974850,
974851,
974852,
974853,
974854,
974855,
974856,
974857,
974858,
974859,
974860,
974862,
974863,
974864,
974865,
974866,
974867,
974868,
974869,
974870,
974871,
974872,
974873,
974874,
974875,
974876,
974877,
974878,
974879,
974880,
974881,
974882,
974883,
974884,
974885,
974886,
974887,
974888,
974889,
974890,
974891,
974892,
974893,
974894,
974895,
974896,
974897,
974898,
974899,
974900,
974901,
974902,
974903,
974904,
974905,
974906,
974907,
974908,
974909,
974910,
974911,
974912,
974913,
974914,
974915,
974916,
974917,
974918,
974919,
974920,
974921,
974922,
974923,
974924,
974925,
974926,
974927,
974928,
974929,
974930,
974931,
974932,
974933,
974934,
974935,
974936,
974937,
974938,
974939,
974940,
974941,
974942,
974943,
974944,
974945,
974946,
974947,
974948,
974949,
974950,
974951,
974952,
974953,
974954,
974955,
974956,
974957,
974958,
974959,
974960,
974961,
974962,
974963,
974964,
974965,
974966,
974967,
974968,
974969,
974970,
974971,
974972,
974973,
974974,
974975,
974976,
974977,
974978,
974979,
974980,
974981,
974982,
974983,
974984,
974985,
974986,
974987,
974988,
974989,
974990,
974991,
974992,
974993,
974994,
974995,
974996,
974997,
974998,
974999,
975000,
975001,
975002,
975003,
975004,
975005,
975006,
975007,
975008,
975009,
975010,
975011,
975012,
975013,
975014,
975015,
975016,
975017,
975018,
975019,
975020,
975021,
975022,
975023,
975024,
975025,
975026,
975027,
975028,
975029,
975030,
975031,
975032,
975033,
975034,
975035,
975036,
975037,
975038,
975039,
975040,
975041,
975042,
975043,
975044,
975045,
975046,
975047,
975048,
975049,
975050,
975051,
975052,
975053,
975054,
975055,
975056,
975057,
975058,
975059,
975060,
975061,
975062,
975063,
975064,
975065,
975066,
975067,
975068,
975069,
975070,
975071,
975072,
975073,
975074,
975075,
975076,
975077,
975078,
975079,
975080,
975081,
975082,
975083,
975084,
975085,
975086,
975087,
975088,
975089,
975090,
975091,
975092,
975093,
975094,
975095,
975096,
975097,
975098,
975099,
975100,
975101,
975102,
975104,
975105,
975106,
975107,
975108,
975109,
975110,
975111,
975112,
975113,
975114,
975115,
975116,
975117,
975118,
975119,
975120,
975121,
975122,
975123,
975124,
975125,
975126,
975127,
975128,
975129,
975130,
975131,
975132,
975133,
975134,
975135,
975136,
975137,
975138,
975139,
975140,
975141,
975142,
975143,
975144,
975145,
975146,
975147,
975148,
975149,
975150,
975151,
975152,
975153,
975154,
975155,
975156,
975157,
975158,
975159,
975160,
975161,
975162,
975163,
975164,
975165,
975166,
975167,
975168,
975169,
975170,
975171,
975172,
975173,
975174,
975175,
975176,
975177,
975178,
975179,
975180,
975181,
975182,
975183,
975184,
975185,
975187,
975188,
975189,
975190,
975191,
975192,
975193,
975194,
975195,
975196,
975197,
975198,
975199,
975200,
975201,
975202,
975203,
975204,
975205,
975206,
975207,
975208,
975209,
975210,
975211,
975212,
975213,
975214,
975215,
975216,
975217,
975218,
975219,
975220,
975221,
975222,
975223,
975224,
975225,
975226,
975227,
975228,
975229,
975230,
975231,
975232,
975233,
975234,
975235,
975236,
975237,
975238,
975239,
975240,
975241,
975242,
975243,
975244,
975245,
975246,
975247,
975248,
975249,
975250,
975251,
975252,
975253,
975254,
975255,
975256,
975257,
975258,
975259,
975260,
975261,
975262,
975263,
975264,
975265,
975266,
975267,
975268,
975269,
975270,
975271,
975272,
975273,
975274,
975275,
975276,
975277,
975278,
975279,
975280,
975281,
975282,
975283,
975284,
975285,
975286,
975287,
975289,
975290,
975291,
975292,
975293,
975294,
975295,
975296,
975297,
975298,
975299,
975300,
975301,
975302,
975303,
975304,
975305,
975306,
975307,
975308,
975309,
975310,
975311,
975312,
975313,
975314,
975315,
975316,
975317,
975318,
975319,
975320,
975321,
975322,
975323,
975324,
975325,
975326,
975327,
975328,
975329,
975330,
975331,
975332,
975333,
975334,
975335,
975336,
975337,
975338,
975339,
975340,
975341,
975342,
975343,
975344,
975345,
975346,
975347,
975348,
975349,
975350,
975351,
975352,
975353,
975354,
975355,
975356,
975357,
975358,
975359,
975360,
975361,
975362,
975363,
975364,
975365,
975366,
975367,
975368,
975369,
975370,
975371,
975372,
975373,
975374,
975375,
975376,
975377,
975378,
975379,
975380,
975381,
975382,
975383,
975384,
975385,
975386,
975387,
975388,
975389,
975390,
975391,
975392,
975393,
975394,
975395,
975396,
975397,
975398,
975399,
975400,
975401,
975402,
975403,
975404,
975405,
975406,
975407,
975408,
975409,
975410,
975411,
975412,
975413,
975414,
975415,
975416,
975417,
975418,
975419,
975420,
975421,
975422,
975423,
975424,
975425,
975426,
975427,
975428,
975429,
975430,
975431,
975432,
975433,
975434,
975435,
975436,
975437,
975438,
975439,
975440,
975441,
975442,
975443,
975444,
975445,
975446,
975447,
975448,
975449,
975450,
975451,
975452,
975453,
975454,
975455,
975456,
975457,
975458,
975459,
975460,
975461,
975462,
975463,
975464,
975465,
975466,
975467,
975468,
975469,
975470,
975471,
975472,
975473,
975474,
975475,
975476,
975477,
975478,
975479,
975480,
975481,
975482,
975483,
975484,
975485,
975486,
975487,
975488,
975489,
975490,
975491,
975492,
975493,
975494,
975495,
975496,
975497,
975498,
975499,
975500,
975501,
975502,
975503,
975504,
975505,
975506,
975507,
975508,
975509,
975510,
975511,
975512,
975513,
975514,
975515,
975516,
975517,
975518,
975519,
975520,
975521,
975522,
975523,
975524,
975525,
975526,
975527,
975528,
975529,
975530,
975531,
975532,
975533,
975534,
975535,
975536,
975537,
975538,
975539,
975540,
975541,
975542,
975543,
975544,
975545,
975546,
975547,
975548,
975549,
975550,
975551,
975552,
975553,
975554,
975555,
975556,
975557,
975558,
975559,
975560,
975561,
975562,
975563,
975564,
975565,
975566,
975567,
975568,
975569,
975570,
975571,
975572,
975573,
975574,
975575,
975576,
975577,
975578,
975579,
975580,
975581,
975582,
975583,
975584,
975585,
975586,
975587,
975588,
975589,
975590,
975591,
975592,
975593,
975594,
975595,
975596,
975597,
975598,
975599,
975600,
975601,
975602,
975603,
975604,
975605,
975606,
975607,
975608,
975609,
975610,
975611,
975612,
975613,
975614,
975615,
975616,
975617,
975618,
975619,
975620,
975621,
975622,
975623,
975624,
975625,
975626,
975627,
975628,
975629,
975630,
975631,
975632,
975633,
975634,
975635,
975636,
975637,
975638,
975639,
975640,
975641,
975642,
975643,
975644,
975645,
975646,
975647,
975648,
718102,
724810,
724852,
724871,
724886,
724915,
724916,
724918,
724928,
724973,
724994,
725017,
725038,
725054,
725106,
725108,
725113,
725140,
725141,
725163,
725172,
725179,
725181,
725194,
725195,
725199,
725222,
725226,
725257,
725285,
725303,
725377,
725390,
725392,
725404,
725409,
725420,
725421,
725424,
725434,
725443,
725447,
725454,
725463,
725467,
725468,
725472,
725473,
725474,
725481,
725514,
725518,
725545,
725589,
725606,
725676,
725711,
725722,
725724,
725737,
725738,
725756,
725758,
725760,
725788,
725795,
725800,
725823,
725834,
725845,
725855,
725860,
725879,
725890,
725897,
725905,
725912,
725946,
725953,
725970,
725986,
726012,
726016,
726022,
726031,
726046,
726050,
726070,
726081,
726082,
726083,
726084,
726102,
726105,
726149,
726151,
726152,
726156,
726165,
726167,
726170,
726171,
726177,
726182,
726185,
726193,
726194,
726197,
726204,
726206,
726207,
726211,
726212,
726215,
726219,
726221,
726239,
726240,
726243,
726245,
726257,
726262,
726263,
726265,
726268,
726270,
726272,
726285,
726293,
726301,
726308,
726312,
726315,
726317,
726318,
726325,
726326,
726331,
726333,
726336,
726338,
726339,
726344,
726347,
726348,
726350,
726354,
726355,
726356,
726359,
726367,
726368,
726370,
726372,
726373,
726374,
726383,
726386,
726396,
726399,
726400,
726402,
726411,
726416,
726418,
726420,
726424,
726425,
726427,
726428,
726434,
726442,
726443,
726445,
726446,
726449,
726450,
726451,
726455,
726456,
726457,
726458,
726459,
726460,
726463,
726464,
726466,
726467,
726469,
726470,
726471,
726472,
726473,
726474,
726475,
726476,
726477,
726478,
726479,
726480,
726481,
726484,
726485,
726486,
726487,
726489,
726490,
726491,
726492,
726493,
726494,
726495,
726496,
726497,
726498,
726499,
726500,
726501,
726502,
726503,
726504,
726505,
726506,
726507,
726508,
726509,
726510,
726511,
726512,
726513,
726514,
726515,
726516,
726517,
726518,
726519,
726520,
726521,
726522,
726523,
726524,
726525,
726526,
726527,
726528,
726529,
726530,
726531,
726534,
726535,
726536,
726537,
726538,
726539,
726540,
726541,
726542,
726543,
726544,
726545,
726546,
726547,
726548,
726549,
726550,
726551,
726553,
726554,
726555,
726556,
726557,
726558,
726559,
726560,
726561,
726562,
726563,
726564,
726565,
726566,
726567,
726568,
726569,
726570,
726571,
726572,
726573,
726575,
726576,
726577,
726581,
726582,
726583,
726584,
726585,
726586,
726587,
726589,
726590,
726591,
726592,
726593,
726594,
726595,
726596,
726597,
726598,
726599,
726600,
726601,
726602,
726603,
726604,
726605,
726606,
726607,
726609,
726610,
726612,
726613,
726614,
726615,
726616,
726617,
726618,
726619,
726620,
726621,
726622,
726623,
726624,
726625,
726626,
726627,
726628,
726629,
726630,
726631,
726632,
726633,
726634,
726635,
726636,
726637,
726638,
726639,
726640,
726641,
726642,
726643,
726644,
726645,
726646,
726647,
726648,
726649,
726650,
726651,
726652,
726653,
726654,
726655,
726656,
726657,
726658,
726660,
726661,
726662,
726663,
726664,
726665,
726666,
726667,
726668,
726669,
726670,
726671,
726672,
726673,
726674,
726675,
726676,
726677,
726678,
726679,
726680,
726681,
726683,
726684,
726685,
726686,
726687,
726688,
726690,
726692,
726693,
726694,
726695,
726696,
726697,
726698,
726699,
726700,
726701,
726702,
726704,
726705,
726706,
726707,
726708,
726709,
726710,
726711,
726712,
726713,
726714,
726715,
726716,
726717,
726718,
726719,
726720,
726721,
726722,
726723,
726724,
726725,
726726,
726727,
726728,
726729,
726730,
726731,
726732,
726733,
726734,
726735,
726736,
726737,
726738,
726739,
726740,
726741,
726742,
726743,
726744,
726745,
726746,
726748,
726749,
726750,
726751,
726752,
726753,
726754,
726755,
726756,
726757,
726758,
726759,
726760,
726761,
726762,
726763,
726765,
726767,
726768,
726769,
726770,
726771,
726772,
726773,
726774,
726775,
726776,
726777,
726778,
726779,
726780,
726781,
726782,
726784,
726785,
726786,
726788,
726789,
726790,
726791,
726792,
726794,
726795,
726796,
726797,
726798,
726799,
726800,
726802,
726803,
726804,
726805,
726806,
726807,
726809,
726810,
726811,
726812,
726813,
726814,
726815,
726816,
726817,
726818,
726819,
726820,
726821,
726822,
726823,
726824,
726825,
726826,
726828,
726830,
726831,
726832,
726833,
726834,
726835,
726836,
726837,
726838,
726839,
726840,
726841,
726842,
726843,
726844,
726845,
726846,
726847,
726848,
726849,
726850,
726851,
726852,
726853,
726854,
726855,
726856,
726857,
726858,
726859,
726860,
726861,
726862,
726863,
726864,
726865,
726866,
726867,
726868,
726869,
726870,
726871,
726872,
726873,
726874,
726875,
726876,
726877,
726878,
726879,
726880,
726881,
726882,
726883,
726884,
726885,
726886,
726887,
726888,
726889,
726890,
726891,
726892,
726893,
726894,
726895,
726896,
726897,
726898,
726899,
726900,
726901,
726902,
726903,
726904,
726905,
726906,
726907,
726908,
726909,
726910,
726911,
726912,
726913,
726914,
726915,
726916,
726917,
726918,
726919,
726920,
726921,
726922,
726923,
726924,
726925,
726926,
726927,
726928,
726929,
726930,
726931,
726932,
726933,
726934,
726935,
726936,
726937,
726938,
726939,
726940,
726941,
726942,
726943,
726944,
726945,
726946,
726947,
726948,
726949,
726950,
726951,
726952,
726953,
726954,
726955,
726956,
726957,
726958,
726959,
726960,
726961,
726962,
726963,
726964,
726965,
726966,
726967,
726968,
726969,
726970,
726971,
726972,
726973,
726974,
726975,
726976,
726977,
726978,
726979,
726980,
726981,
726982,
726983,
726984,
726985,
726986,
726987,
726988,
726989,
726990,
726991,
726992,
726993,
726994,
726995,
726996,
798001,
798002]

In [ ]:
################################################################# ROUGH #################################################################

In [ ]:
# k=0
# i=0
# data_lst = []
# while k==0:
#   try:
#     text_list = [td.get_text().strip() for td in rows[i+17].find_all('td')[1:8]]
#     if text_list[0]=='B1) Institutions':
#       append_flag = True
#     if append_flag:
#       data_lst.append(text_list)
#     i+=1
#   except:
#     append_flag = False
#     k+=1

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType

# Initialize SparkSession
spark = SparkSession.builder.appName("Empty DataFrame Example").getOrCreate()

# Define schema
schema = StructType([
    StructField("CategoryNameoftheShareholders", StringType(), True),
    StructField("NoOfShareholder", StringType(), True),
    StructField("NoOfFullyPaidShares", StringType(), True),
    StructField("Blank_1", StringType(), True),
    StructField("Blank_2", StringType(), True),
    StructField("TotalNoSharesHeld", StringType(), True),
    StructField("ShareholdingPerc", StringType(), True)
])

# Create an empty DataFrame
empty_df = spark.createDataFrame([], schema)

# Show the empty DataFrame
empty_df.show()


In [ ]:
# url = f"https://www.bseindia.com/corporates/shpPublicShareholder.aspx?scripcd=500033&qtrid=121.00&QtrName=March%202024"
Number = 500033
Qtr = "March"
Yer = "2024"
url = f"https://www.bseindia.com/corporates/shpPublicShareholder.aspx?scripcd={Number}&qtrid=121.00&QtrName={Qtr}%20{Yer}"

# Define headers to mimic a real browser request
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

# Define cookies if required
cookies = {
    "cookie_name": "cookie_value"
}

# Send a GET request to the URL with headers and cookies
response = requests.get(url, headers=headers, cookies=cookies)

soup = BeautifulSoup(response.content, "html.parser")

rows = soup.find_all('tr')

for td in rows[7].find_all('td'):
  title = td.get_text().strip()

data_lst = []
for i in range(17, len(rows)):
  try:
    text_list = [td.get_text().strip() for td in rows[i].find_all('td')[1:8]]
    if text_list[0]=='B1) Institutions':
      append_flag = True
    if append_flag:
      data_lst.append(text_list)
    i+=1
  except IndexError:
    append_flag = False

# for i in data_lst:
#   print(len(i), i)

columns = ['CategoryNameoftheShareholders', 'NoOfShareholder', 'NoOfFullyPaidShares', 'Blank_1', 'Blank_2', 'TotalNoSharesHeld', 'ShareholdingPerc']
pandas_df = pd.DataFrame(data_lst, columns=columns)

# Replace null values and empty spaces with 0
# pandas_df.replace(to_replace=['', ' ', None, pd.NA], value=0, inplace=True)

# Convert data types
pandas_df['NoOfShareholder'] = pandas_df['NoOfShareholder'].astype(int)
pandas_df['ShareholdingPerc'] = pandas_df['ShareholdingPerc'].astype(float)

# Filter rows where NoOfShareholder is equal to 1
pandas_df = pandas_df[pandas_df['NoOfShareholder'] == 1]

# Drop the original Blank_1 and Blank_2 columns
pandas_df.drop(columns=['Blank_1', 'Blank_2', 'NoOfShareholder', 'TotalNoSharesHeld'], inplace=True)
pandas_df['Number'] = Number
pandas_df['Title'] = title
pandas_df['Qtr'] = Qtr
pandas_df['Year'] = Yer
pandas_df